In [1]:
# [0-1] 패키지 설치
!pip install -q onnxruntime-gpu==1.20.1 filterpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 9.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.5/291.5 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.9 MB/s eta 0:00:00


In [2]:
# [0-2] 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

# Google Drive (MyDrive)
# └── CONTEST
#     └── cj_box_sizing
#       └── assignment1
#           └── dataset/
#               ├── train/       <-- Train MP4 영상 100개
#               ├── train_label.json    <-- Train 정답 json 파일
#               └── test/        <-- Test MP4 영상 50개

Mounted at /content/drive


In [3]:
# [0-3] 임포트
import os, shutil, glob, math, json, pickle, random
import cv2
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import onnxruntime as ort
from collections import defaultdict
from filterpy.kalman import KalmanFilter
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm

In [16]:
# ==============================================================================
# [1-1] 경로 설정
# ==============================================================================
PROJECT_DIR = '/content/drive/MyDrive/CONTEST/cj_box_sizing'

ONNX_PATH        = '/content/drive/MyDrive/CONTEST/cj_box_sizing/checkpoints/yolo26l_v6_manual_augmentation.onnx'  # Box Detect YOLO Model Path
RAIL_ONNX_PATH   = '/content/drive/MyDrive/CONTEST/cj_box_sizing/YOLOV8_for_RAIL_DETECT_runs/rail_seg_yolov8s_LabelStudio/weights/best.onnx' # RAIL Detect YOLO v8 Model Path
DA3_PATH         = '/content/drive/MyDrive/CONTEST/cj_box_sizing/depth_anything_v3/DA3METRIC-LARGE.onnx' # depth_anything_v3 Model Path
MAPPING_XLSX_PATH = '/content/drive/MyDrive/CONTEST/cj_box_sizing/train_label_mapping.xlsx'

TEST_VIDEO_DIR   = os.path.join(PROJECT_DIR, 'assignment1', 'dataset', 'test')
TRAIN_VIDEO_DIR  = os.path.join(PROJECT_DIR, 'assignment1', 'dataset', 'train')
TRAIN_LABEL_PATH = os.path.join(PROJECT_DIR, 'assignment1', 'dataset', 'train_label.json')

WORK_DIR    = '/content/work'
DATASET_OUT = os.path.join(WORK_DIR, 'box_dataset_with_yolo_version6')
CKPT_OUT    = os.path.join(PROJECT_DIR, 'checkpoints_submit')
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(DATASET_OUT, exist_ok=True)
os.makedirs(CKPT_OUT, exist_ok=True)

CROP_SIZE     = 128   # Box Regressor에 들어가는 입력
RAIL_WIDTH_CM = 62.3

# DA3 없으면 다운로드 (cj_box_sizing/depth_anything_v3/ 에 저장됨)
if not os.path.exists(DA3_PATH):
    print("⬇️ DA3 다운로드 (~731MB)...")
    import urllib.request
    os.makedirs(os.path.dirname(DA3_PATH), exist_ok=True)
    urllib.request.urlretrieve(
        "https://huggingface.co/TillBeemelmanns/Depth-Anything-V3-ONNX/resolve/main/DA3METRIC-LARGE.onnx",
        DA3_PATH)

for p, name in [(TRAIN_VIDEO_DIR,'TRAIN'), (TRAIN_LABEL_PATH,'LABEL'),
                (TEST_VIDEO_DIR,'TEST'), (ONNX_PATH,'BOX_ONNX'),
                (RAIL_ONNX_PATH,'RAIL_ONNX'), (DA3_PATH,'DA3')]:
    print(f"{name:9} {'✅' if os.path.exists(p) else '❌'}  {p}")

with open(TRAIN_LABEL_PATH) as f:   # ← LABEL_PATH → TRAIN_LABEL_PATH
    label_data = json.load(f)
label_map = {v['video_id']: v for v in label_data['videos']}

SENSOR_W   = float(label_data['videos'][0]['camera']['sensor_width_mm'])
FOCAL_MEAN = float(np.mean([v['camera']['focal_length_mm'] for v in label_data['videos']]))
print(f"\nlabel_map {len(label_map)}개  |  sensor_w={SENSOR_W:.2f}  focal 평균={FOCAL_MEAN:.3f}")

train_videos = sorted(glob.glob(os.path.join(TRAIN_VIDEO_DIR, '**', '*.mp4'), recursive=True))   # ← TRAIN_DIR → TRAIN_VIDEO_DIR
print(f"train 영상 {len(train_videos)}개")

TRAIN     ✅  /content/drive/MyDrive/CONTEST/cj_box_sizing/assignment1/dataset/train
LABEL     ✅  /content/drive/MyDrive/CONTEST/cj_box_sizing/assignment1/dataset/train_label.json
TEST      ✅  /content/drive/MyDrive/CONTEST/cj_box_sizing/assignment1/dataset/test
BOX_ONNX  ✅  /content/drive/MyDrive/CONTEST/cj_box_sizing/checkpoints/yolo26l_v6_manual_augmentation.onnx
RAIL_ONNX ✅  /content/drive/MyDrive/CONTEST/cj_box_sizing/YOLOV8_for_RAIL_DETECT_runs/rail_seg_yolov8s_LabelStudio/weights/best.onnx
DA3       ✅  /content/drive/MyDrive/CONTEST/cj_box_sizing/depth_anything_v3/DA3METRIC-LARGE.onnx

label_map 100개  |  sensor_w=5.79  focal 평균=10.678
train 영상 100개


## STAGE 1. Box Detect

In [6]:
# ==============================================================================
# [2-1] 상자 검출 — 하이퍼파라미터 + ONNX 세션 + 검출 함수
# ==============================================================================
MOTION_THRESHOLD    = 44
MOTION_RATIO_THRESH = 0.0294
ROI_PAD             = 20

BASE_MAX_W_RATIO     = 0.70
BASE_MAX_H_RATIO     = 0.70
RELAXED_MAX_W_RATIO  = 0.90
RELAXED_MAX_H_RATIO  = 0.90

BASE_MIN_CONF     = 0.6
RELAXED_MIN_CONF  = 0.85
DUPLICATE_OVERLAP_THRESH = 0.90

MAX_W_RATIO = BASE_MAX_W_RATIO
MAX_H_RATIO = BASE_MAX_H_RATIO

LABEL_INTERVAL_SEC  = 0.1
CONF_THRESH         = 0.25
IOU_THRESH          = 0.45

BT_HIGH_THRESH  = 0.50
BT_LOW_THRESH   = 0.25
BT_IOU_THRESH_1 = 0.30
BT_IOU_THRESH_2 = 0.20
BT_MAX_MISS     = 45

MERGE_IOU_THRESH    = 0.20
MERGE_MAX_FRAME_GAP = 30

POST_NMS_THRESH = 0.60

LANE_TOLERANCE_RATIO = 0.6
DEPTH_DIST_RATIO     = 1.5

FORWARD_ONLY_TOLERANCE_RATIO = 0.3

# edge-aware 병합(부분 가림 재검출 처리) 파라미터
PARTIAL_AREA_RATIO         = 0.5
EDGE_TOLERANCE_RATIO       = 0.35
CROSS_AXIS_TOLERANCE_RATIO = 0.7

# 극단적으로 짧은(노이즈성) 트랙 처리 파라미터
MIN_TRACK_FRAMES_FOR_MERGE = 2
MIN_TRACK_FRAMES_FOR_COUNT = 2

print("🔍 상자 검출 onnxruntime 세션 로드 중...")
providers   = ['CUDAExecutionProvider', 'CPUExecutionProvider']
session     = ort.InferenceSession(ONNX_PATH, providers=providers)
input_name  = session.get_inputs()[0].name
input_shape = session.get_inputs()[0].shape
print(f"✅ 로드 완료  |  input: {input_name}  shape: {input_shape}")
print(f"   사용 provider: {session.get_providers()}")


def preprocess(img_bgr, imgsz=640):
    h, w  = img_bgr.shape[:2]
    scale = imgsz / max(h, w)
    nh, nw = int(h * scale), int(w * scale)
    resized = cv2.resize(img_bgr, (nw, nh))

    canvas   = np.full((imgsz, imgsz, 3), 114, dtype=np.uint8)
    pad_top  = (imgsz - nh) // 2
    pad_left = (imgsz - nw) // 2
    canvas[pad_top:pad_top+nh, pad_left:pad_left+nw] = resized

    tensor = canvas[:, :, ::-1].astype(np.float32) / 255.0
    tensor = tensor.transpose(2, 0, 1)[np.newaxis]
    return tensor, scale, pad_top, pad_left


def postprocess(outputs, scale, pad_top, pad_left,
                conf_thresh=0.25,
                crop_h=640, crop_w=640,
                crop_bgr=None):
    """
    1차: BASE(0.7) 이내이면서 conf ≥ BASE_MIN_CONF(0.6)인 detection만 통과 (is_relaxed=False)
    2차: BASE 초과 ~ RELAXED(0.9) 이내이면서 conf ≥ RELAXED_MIN_CONF(0.85)인
         detection만 재검토 후보로 통과 (is_relaxed=True)
    반환: (x1,y1,x2,y2,conf,is_relaxed) 6-튜플
    """
    pred = outputs[0][0]
    kept = []

    for row in pred:
        x1, y1, x2, y2, conf, cls_id = row
        if conf < conf_thresh:
            continue

        x1 = (x1 - pad_left) / scale
        y1 = (y1 - pad_top)  / scale
        x2 = (x2 - pad_left) / scale
        y2 = (y2 - pad_top)  / scale
        bx1, by1, bx2, by2 = map(int, [x1, y1, x2, y2])

        w, h = (bx2 - bx1), (by2 - by1)

        if w <= crop_w * BASE_MAX_W_RATIO and h <= crop_h * BASE_MAX_H_RATIO:
            if conf >= BASE_MIN_CONF:
                kept.append((bx1, by1, bx2, by2, float(conf), False))
        elif (w <= crop_w * RELAXED_MAX_W_RATIO and h <= crop_h * RELAXED_MAX_H_RATIO
              and conf >= RELAXED_MIN_CONF):
            kept.append((bx1, by1, bx2, by2, float(conf), True))

    return kept


def post_nms(dets, iou_thresh=POST_NMS_THRESH):
    if len(dets) == 0:
        return dets
    dets_sorted = sorted(dets, key=lambda d: d[4], reverse=True)
    keep = []
    while dets_sorted:
        best = dets_sorted.pop(0)
        keep.append(best)
        remaining = []
        for d in dets_sorted:
            ax1, ay1, ax2, ay2 = best[:4]
            bx1, by1, bx2, by2 = d[:4]
            ix1 = max(ax1, bx1); iy1 = max(ay1, by1)
            ix2 = min(ax2, bx2); iy2 = min(ay2, by2)
            inter = max(0, ix2-ix1) * max(0, iy2-iy1)
            if inter == 0:
                remaining.append(d)
                continue
            ua  = (ax2-ax1)*(ay2-ay1) + (bx2-bx1)*(by2-by1) - inter
            iou = inter / ua
            if iou < iou_thresh:
                remaining.append(d)
        dets_sorted = remaining
    return keep


def ort_detect(crop_bgr, post_nms_thresh=POST_NMS_THRESH):
    if crop_bgr is None or crop_bgr.size == 0:
        return []
    crop_h, crop_w = crop_bgr.shape[:2]
    tensor, scale, pad_top, pad_left = preprocess(crop_bgr)
    outputs = session.run(None, {input_name: tensor})
    dets = postprocess(
        outputs, scale, pad_top, pad_left,
        conf_thresh = CONF_THRESH,
        crop_h      = crop_h,
        crop_w      = crop_w,
        crop_bgr    = crop_bgr,
    )
    return post_nms(dets, iou_thresh=post_nms_thresh)

🔍 상자 검출 onnxruntime 세션 로드 중...
✅ 로드 완료  |  input: images  shape: [1, 3, 640, 640]
   사용 provider: ['CUDAExecutionProvider', 'CPUExecutionProvider']


## STAGE 2. Tracking

In [7]:
# ==============================================================================
# [3-1] 트래킹/ROI
# ==============================================================================

def extract_frames(video_path, interval_sec=LABEL_INTERVAL_SEC):
    cap  = cv2.VideoCapture(video_path)
    fps  = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step = max(int(round(fps * interval_sec)), 1)
    out, idx = [], 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if idx % step == 0:
            out.append((frame, idx / fps))
        idx += 1
    cap.release()
    return out, fps


def get_motion_roi_bbox(video_path,
                        motion_threshold    = MOTION_THRESHOLD,
                        motion_ratio_thresh = MOTION_RATIO_THRESH,
                        interval_sec        = 0.5,
                        pad                 = ROI_PAD,
                        min_sample_frames   = 20):
    """★ 폴백 전용 (auto_detect_roi / get_roi_for_test_video 실패 시에만 사용)"""
    cap   = cv2.VideoCapture(video_path)
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    frame_step = max(1, min(int(fps * interval_sec), total // min_sample_frames))

    extra = set()
    for i in range(min(5, total)):
        extra.add(i)
    for i in range(max(0, total - 5), total):
        extra.add(i)

    sample_indices = np.array(
        sorted(set(np.arange(0, total, frame_step, dtype=int).tolist()) | extra)
    )

    grays = []
    for idx in sample_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, fr = cap.read()
        if ret:
            grays.append(cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY))
    cap.release()

    if len(grays) < 3:
        return None

    stack        = np.stack(grays, 0).astype(np.float32)
    bg           = np.median(stack, 0)
    motion_count = np.zeros((H, W), np.float32)
    for g in grays:
        motion_count += (np.abs(g.astype(np.float32) - bg) > motion_threshold)
    motion_ratio = motion_count / len(grays)

    duration_sec    = total / max(fps, 1)
    adaptive_thresh = motion_ratio_thresh
    if duration_sec < 5:
        adaptive_thresh = motion_ratio_thresh * 0.4
    elif duration_sec < 10:
        adaptive_thresh = motion_ratio_thresh * 0.7

    mask = (motion_ratio >= adaptive_thresh).astype(np.uint8) * 255
    k    = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k)

    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return (0, 0, W, H)

    bx, by, bw, bh = cv2.boundingRect(max(cnts, key=cv2.contourArea))
    return (max(0, bx - pad), max(0, by - pad),
            min(W, bx + bw + pad), min(H, by + bh + pad))


GLOBAL_MOTION_THRESH = MOTION_THRESHOLD
GLOBAL_RATIO_THRESH  = MOTION_RATIO_THRESH

def get_roi_for_test_video(video_path,
                           motion_threshold=GLOBAL_MOTION_THRESH,
                           motion_ratio_thresh=GLOBAL_RATIO_THRESH):
    cap     = cv2.VideoCapture(video_path)
    fps     = cap.get(cv2.CAP_PROP_FPS)
    total_f = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    orig_w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    min_sample_frames = 20
    frame_step = max(1, min(int(fps * 0.5), max(total_f // min_sample_frames, 1)))
    extra = set(range(min(5, total_f))) | set(range(max(0, total_f - 5), total_f))
    sample_indices = np.array(
        sorted(set(np.arange(0, total_f, frame_step, dtype=int).tolist()) | extra)
    )

    sampled_frames = []
    for idx in sample_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if ret:
            sampled_frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
    cap.release()

    if len(sampled_frames) < 3:
        return None

    stack     = np.stack(sampled_frames, axis=0).astype(np.float32)
    bg_median = np.median(stack, axis=0)
    motion_count = np.zeros((orig_h, orig_w), dtype=np.float32)
    for frame_gray in sampled_frames:
        diff = np.abs(frame_gray.astype(np.float32) - bg_median)
        motion_count += (diff > motion_threshold).astype(np.float32)
    motion_ratio = motion_count / len(sampled_frames)

    duration_sec    = total_f / max(fps, 1)
    adaptive_thresh = motion_ratio_thresh
    if duration_sec < 5:
        adaptive_thresh = motion_ratio_thresh * 0.4
    elif duration_sec < 10:
        adaptive_thresh = motion_ratio_thresh * 0.7

    motion_mask = (motion_ratio >= adaptive_thresh).astype(np.uint8) * 255
    kernel      = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    motion_mask = cv2.morphologyEx(motion_mask, cv2.MORPH_CLOSE, kernel)
    motion_mask = cv2.morphologyEx(motion_mask, cv2.MORPH_OPEN,  kernel)

    contours, _ = cv2.findContours(motion_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return (0, 0, orig_w, orig_h)

    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    bx, by, bw, bh = cv2.boundingRect(contours[0])
    pad = 20
    return (max(0, bx-pad), max(0, by-pad),
            min(orig_w, bx+bw+pad), min(orig_h, by+bh+pad))


def get_roi_for_train_video(video_path,
                            motion_threshold=GLOBAL_MOTION_THRESH,
                            motion_ratio_thresh=GLOBAL_RATIO_THRESH):
    """test용 get_roi_for_test_video와 완전히 동일한 로직 (train 경로 대응 wrapper)
       — FINAL TRACKING CODE에는 없지만, 로직은 100% 동일하게 유지"""
    return get_roi_for_test_video(video_path, motion_threshold, motion_ratio_thresh)


# ==============================================================================
# 컨베이어 속도 추정
# ==============================================================================
def estimate_conveyor_speed(all_tracks):
    vx_list, vy_list = [], []
    for tid, hist in all_tracks.items():
        if len(hist) < 3:
            continue
        frames = [h[0] for h in hist]
        bboxes = [h[1] for h in hist]
        for k in range(1, len(frames)):
            dt = frames[k] - frames[k-1]
            if dt <= 0 or dt > 5:
                continue
            cx_prev = (bboxes[k-1][0] + bboxes[k-1][2]) / 2
            cy_prev = (bboxes[k-1][1] + bboxes[k-1][3]) / 2
            cx_cur  = (bboxes[k][0]   + bboxes[k][2])   / 2
            cy_cur  = (bboxes[k][1]   + bboxes[k][3])   / 2
            vx_list.append((cx_cur - cx_prev) / dt)
            vy_list.append((cy_cur - cy_prev) / dt)

    if not vx_list:
        return 0.0, 0.0
    return float(np.median(vx_list)), float(np.median(vy_list))


def compute_dynamic_params(vx, vy,
                           frame_interval_sec=LABEL_INTERVAL_SEC,
                           bbox_avg_px=150):
    speed_px_per_frame = np.sqrt(vx**2 + vy**2)

    if speed_px_per_frame < 1.0:
        return 30, 35

    frames_per_box = bbox_avg_px / speed_px_per_frame
    max_miss  = int(np.clip(frames_per_box * 1.5, 5, BT_MAX_MISS))
    merge_gap = int(np.clip(frames_per_box * 2.0, 5, MERGE_MAX_FRAME_GAP))

    return int(max_miss), int(merge_gap)


# ==============================================================================
# 공간 순서 분석
# ==============================================================================
def build_spatial_order(all_tracks, conveyor_vx=0.0, conveyor_vy=0.0):
    track_profiles = {}
    for tid, hist in all_tracks.items():
        if not hist:
            continue
        frames  = [h[0] for h in hist]
        bboxes  = [h[1] for h in hist]

        cx_vals = [(b[0]+b[2])/2 for b in bboxes]
        cy_vals = [(b[1]+b[3])/2 for b in bboxes]
        w_vals  = [b[2]-b[0] for b in bboxes]
        h_vals  = [b[3]-b[1] for b in bboxes]

        cx_rep = float(np.median(cx_vals))

        first_idx  = int(np.argmin(frames))
        cy_entry   = cy_vals[first_idx]

        track_profiles[tid] = {
            'cx'         : cx_rep,
            'cy_entry'   : cy_entry,
            'first_frame': min(frames),
            'last_frame' : max(frames),
            'avg_w'      : float(np.median(w_vals)),
            'avg_h'      : float(np.median(h_vals)),
        }

    return track_profiles


def is_same_lane(prof_a, prof_b, lane_tolerance_ratio=LANE_TOLERANCE_RATIO):
    avg_w   = (prof_a['avg_w'] + prof_b['avg_w']) / 2
    cx_diff = abs(prof_a['cx'] - prof_b['cx'])
    return cx_diff < avg_w * lane_tolerance_ratio


def depth_order_consistent(earlier_prof, later_prof, conveyor_vy=0.0):
    if abs(conveyor_vy) < 0.5:
        return True

    cy_diff = later_prof['cy_entry'] - earlier_prof['cy_entry']

    if conveyor_vy > 0:
        tolerance = (earlier_prof['avg_h'] + later_prof['avg_h']) / 2
        return cy_diff > -tolerance
    else:
        tolerance = (earlier_prof['avg_h'] + later_prof['avg_h']) / 2
        return cy_diff < tolerance


# ==============================================================================
# KalmanBoxTracker + 매칭 유틸
# ==============================================================================
class KalmanBoxTracker:
    count = 0

    def __init__(self, bbox, conf, init_vx=0.0, init_vy=0.0):
        self.kf = KalmanFilter(dim_x=7, dim_z=4)
        self.kf.F = np.array([
            [1,0,0,0,1,0,0],
            [0,1,0,0,0,1,0],
            [0,0,1,0,0,0,1],
            [0,0,0,1,0,0,0],
            [0,0,0,0,1,0,0],
            [0,0,0,0,0,1,0],
            [0,0,0,0,0,0,1],
        ], dtype=float)
        self.kf.H = np.array([
            [1,0,0,0,0,0,0],
            [0,1,0,0,0,0,0],
            [0,0,1,0,0,0,0],
            [0,0,0,1,0,0,0],
        ], dtype=float)
        self.kf.R[2:, 2:] *= 10.0
        self.kf.P[4:, 4:] *= 1000.0
        self.kf.P         *= 10.0
        self.kf.Q[-1,-1]  *= 0.01
        self.kf.Q[4:,4:]  *= 0.01
        self.kf.x[:4]      = self._bbox_to_z(bbox)

        self.kf.x[4, 0] = init_vx
        self.kf.x[5, 0] = init_vy

        self.id      = KalmanBoxTracker.count
        KalmanBoxTracker.count += 1
        self.hits    = 1
        self.miss    = 0
        self.conf    = conf
        self.history = []

        self.last_matched_bbox = tuple(map(int, bbox))

    def _bbox_to_z(self, bbox):
        x1, y1, x2, y2 = bbox
        cx    = (x1 + x2) / 2.0
        cy    = (y1 + y2) / 2.0
        area  = (x2 - x1) * (y2 - y1)
        ratio = (x2 - x1) / float(y2 - y1 + 1e-6)
        return np.array([[cx],[cy],[area],[ratio]], dtype=float)

    def _z_to_bbox(self):
        cx, cy, area, ratio = (
            self.kf.x[0,0], self.kf.x[1,0],
            self.kf.x[2,0], self.kf.x[3,0]
        )
        area  = max(area, 1.0)
        ratio = max(ratio, 0.1)
        w = np.sqrt(area * ratio)
        h = area / w
        return np.array([cx-w/2, cy-h/2, cx+w/2, cy+h/2])

    def predict(self):
        if self.kf.x[2,0] + self.kf.x[6,0] <= 0:
            self.kf.x[6,0] = 0.0
        self.kf.predict()
        self.miss += 1
        return self._z_to_bbox()

    def update(self, bbox, conf, frame_idx):
        self.kf.update(self._bbox_to_z(bbox))
        self.hits += 1
        self.miss  = 0
        self.conf  = conf
        self.history.append((frame_idx, tuple(map(int, bbox)), conf))
        self.last_matched_bbox = tuple(map(int, bbox))

    def get_bbox(self):
        return self._z_to_bbox()

    def get_cx(self):
        return float(self.kf.x[0, 0])

    def get_cy(self):
        return float(self.kf.x[1, 0])


def _iou_matrix(trackers, detections):
    iou_mat = np.zeros((len(trackers), len(detections)), dtype=float)
    for i, t in enumerate(trackers):
        tb = t.get_bbox()
        for j, d in enumerate(detections):
            db = d[:4]
            xx1 = max(tb[0], db[0]); yy1 = max(tb[1], db[1])
            xx2 = min(tb[2], db[2]); yy2 = min(tb[3], db[3])
            inter = max(0, xx2-xx1) * max(0, yy2-yy1)
            if inter == 0:
                continue
            ua = ((tb[2]-tb[0])*(tb[3]-tb[1]) +
                  (db[2]-db[0])*(db[3]-db[1]) - inter)
            iou_mat[i,j] = inter / ua
    return iou_mat


def _candidate_overlap_ratio(base_box, candidate_box):
    ax1, ay1, ax2, ay2 = base_box[:4]
    bx1, by1, bx2, by2 = candidate_box[:4]
    ix1 = max(ax1, bx1); iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2); iy2 = min(ay2, by2)
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    if inter == 0:
        return 0.0
    candidate_area = (bx2-bx1) * (by2-by1)
    if candidate_area <= 0:
        return 0.0
    return inter / candidate_area


def _hungarian_match(iou_mat, thresh):
    if iou_mat.size == 0:
        return [], list(range(iou_mat.shape[0])), list(range(iou_mat.shape[1]))

    row_ind, col_ind = linear_sum_assignment(-iou_mat)
    matched, unmatched_t, unmatched_d = [], [], []
    matched_rows, matched_cols = set(), set()

    for r, c in zip(row_ind, col_ind):
        if iou_mat[r,c] >= thresh:
            matched.append((r,c))
            matched_rows.add(r)
            matched_cols.add(c)

    for r in range(iou_mat.shape[0]):
        if r not in matched_rows:
            unmatched_t.append(r)
    for c in range(iou_mat.shape[1]):
        if c not in matched_cols:
            unmatched_d.append(c)

    return matched, unmatched_t, unmatched_d


def _lane_distance_matrix(trackers, detections,
                          lane_tolerance_ratio=LANE_TOLERANCE_RATIO):
    lane_mat = np.full((len(trackers), len(detections)), np.inf)
    for i, t in enumerate(trackers):
        tb   = t.get_bbox()
        tcx  = (tb[0] + tb[2]) / 2
        t_w  = tb[2] - tb[0]
        for j, d in enumerate(detections):
            dcx    = (d[0] + d[2]) / 2
            d_w    = d[2] - d[0]
            avg_w  = (t_w + d_w) / 2
            cx_diff = abs(tcx - dcx)
            if cx_diff < avg_w * lane_tolerance_ratio:
                tcy = (tb[1] + tb[3]) / 2
                dcy = (d[1] + d[3]) / 2
                lane_mat[i, j] = abs(tcy - dcy) / max(avg_w, 1)
    return lane_mat


def _apply_forward_only_mask(trackers, detections, vx, vy,
                             tolerance_ratio=FORWARD_ONLY_TOLERANCE_RATIO):
    n_t, n_d = len(trackers), len(detections)
    mask = np.ones((n_t, n_d), dtype=bool)

    speed = np.sqrt(vx**2 + vy**2)
    if speed < 0.5:
        return mask

    ux, uy = vx / speed, vy / speed

    for i, t in enumerate(trackers):
        lx1, ly1, lx2, ly2 = t.last_matched_bbox
        lcx, lcy = (lx1+lx2)/2, (ly1+ly2)/2
        lw, lh   = (lx2-lx1), (ly2-ly1)

        for j, d in enumerate(detections):
            dx1, dy1, dx2, dy2 = d[:4]
            dcx, dcy = (dx1+dx2)/2, (dy1+dy2)/2
            dw, dh   = (dx2-dx1), (dy2-dy1)

            disp_x, disp_y = dcx - lcx, dcy - lcy
            projection = disp_x * ux + disp_y * uy

            avg_size  = (lw + lh + dw + dh) / 4
            tolerance = avg_size * tolerance_ratio

            if projection < -tolerance:
                mask[i, j] = False

    return mask


# ==============================================================================
# ByteTracker
# ==============================================================================
class ByteTracker:
    def __init__(self,
                 high_thresh  = BT_HIGH_THRESH,
                 low_thresh   = BT_LOW_THRESH,
                 iou_thresh_1 = BT_IOU_THRESH_1,
                 iou_thresh_2 = BT_IOU_THRESH_2,
                 max_miss     = BT_MAX_MISS,
                 conveyor_vx  = 0.0,
                 conveyor_vy  = 0.0,
                 pos_dist_thresh    = DEPTH_DIST_RATIO,
                 lane_tol_ratio     = LANE_TOLERANCE_RATIO,
                 forward_only_tol   = FORWARD_ONLY_TOLERANCE_RATIO,
                 duplicate_overlap_thresh = DUPLICATE_OVERLAP_THRESH):
        self.high_thresh       = high_thresh
        self.low_thresh        = low_thresh
        self.iou_thresh_1      = iou_thresh_1
        self.iou_thresh_2      = iou_thresh_2
        self.max_miss          = max_miss
        self.conveyor_vx       = conveyor_vx
        self.conveyor_vy       = conveyor_vy
        self.pos_dist_thresh   = pos_dist_thresh
        self.lane_tol_ratio    = lane_tol_ratio
        self.forward_only_tol  = forward_only_tol
        self.duplicate_overlap_thresh = duplicate_overlap_thresh
        self.trackers          = []
        self.dead_trackers     = []
        KalmanBoxTracker.count = 0

    def update(self, dets, frame_idx):
        base_dets    = [d for d in dets if not (len(d) > 5 and d[5])]
        relaxed_dets = [d for d in dets if (len(d) > 5 and d[5])]

        filtered_relaxed = []
        for d in relaxed_dets:
            max_overlap = 0.0
            for b in base_dets:
                ov = _candidate_overlap_ratio(b[:4], d[:4])
                if ov > max_overlap:
                    max_overlap = ov
            if max_overlap < self.duplicate_overlap_thresh:
                filtered_relaxed.append(d)

        dets = base_dets + filtered_relaxed

        high_dets = [d for d in dets if d[4] >= self.high_thresh]
        low_dets  = [d for d in dets if self.low_thresh <= d[4] < self.high_thresh]

        for t in self.trackers:
            t.predict()

        iou1 = _iou_matrix(self.trackers, high_dets)
        if len(self.trackers) > 0 and len(high_dets) > 0:
            fwd_mask1 = _apply_forward_only_mask(
                self.trackers, high_dets,
                self.conveyor_vx, self.conveyor_vy, self.forward_only_tol)
            iou1 = np.where(fwd_mask1, iou1, 0.0)

        matched1, unmatched_t1, unmatched_d1 = _hungarian_match(iou1, self.iou_thresh_1)

        for ti, di in matched1:
            self.trackers[ti].update(high_dets[di][:4], high_dets[di][4], frame_idx)

        remain2_t    = [self.trackers[i] for i in unmatched_t1]
        remain2_idxs = list(unmatched_t1)
        iou2         = _iou_matrix(remain2_t, low_dets)
        if len(remain2_t) > 0 and len(low_dets) > 0:
            fwd_mask2 = _apply_forward_only_mask(
                remain2_t, low_dets,
                self.conveyor_vx, self.conveyor_vy, self.forward_only_tol)
            iou2 = np.where(fwd_mask2, iou2, 0.0)

        matched2, unmatched_t2_local, _ = _hungarian_match(iou2, self.iou_thresh_2)

        matched_t2_set = set(remain2_idxs[li] for li, _ in matched2)
        for li, di in matched2:
            remain2_t[li].update(low_dets[di][:4], low_dets[di][4], frame_idx)

        still_unmatched_t = [
            self.trackers[i] for i in unmatched_t1
            if i not in matched_t2_set
        ]
        still_unmatched_d = [
            high_dets[i] for i in unmatched_d1
        ]

        if still_unmatched_t and still_unmatched_d:
            lane_mat = _lane_distance_matrix(
                still_unmatched_t, still_unmatched_d, self.lane_tol_ratio)

            fwd_mask3 = _apply_forward_only_mask(
                still_unmatched_t, still_unmatched_d,
                self.conveyor_vx, self.conveyor_vy, self.forward_only_tol)
            lane_mat = np.where(fwd_mask3, lane_mat, np.inf)

            matched3_local = []
            used_t, used_d = set(), set()

            flat_indices = np.argsort(lane_mat.ravel())
            for idx in flat_indices:
                ti_l = idx // len(still_unmatched_d)
                di_l = idx  % len(still_unmatched_d)
                if lane_mat[ti_l, di_l] == np.inf:
                    break
                if lane_mat[ti_l, di_l] > self.pos_dist_thresh:
                    break
                if ti_l in used_t or di_l in used_d:
                    continue
                matched3_local.append((ti_l, di_l))
                used_t.add(ti_l)
                used_d.add(di_l)

            for ti_l, di_l in matched3_local:
                d = still_unmatched_d[di_l]
                still_unmatched_t[ti_l].update(d[:4], d[4], frame_idx)

            matched3_dets = {id(still_unmatched_d[di_l]) for _, di_l in matched3_local}
            unmatched_d1_final = [
                di for di in unmatched_d1
                if id(high_dets[di]) not in matched3_dets
            ]
        else:
            unmatched_d1_final = list(unmatched_d1)

        for di in unmatched_d1_final:
            d     = high_dets[di]
            new_t = KalmanBoxTracker(d[:4], d[4],
                                     init_vx=self.conveyor_vx,
                                     init_vy=self.conveyor_vy)
            new_t.history.append((frame_idx, tuple(map(int, d[:4])), d[4]))
            self.trackers.append(new_t)

        alive, dead = [], []
        for t in self.trackers:
            if t.miss <= self.max_miss:
                alive.append(t)
            else:
                dead.append(t)
        self.trackers = alive
        self.dead_trackers.extend(dead)

        results = []
        for t in self.trackers:
            if t.miss > 0:
                continue
            bbox = t.get_bbox()
            x1, y1, x2, y2 = map(int, bbox)
            results.append((t.id, x1, y1, x2, y2, t.conf))
        return results

    def get_all_tracks(self):
        all_t = self.trackers + self.dead_trackers
        return {t.id: t.history for t in all_t if t.history}


# ==============================================================================
# Track 병합 — ★ edge-aware 부분가림 처리 + range_overlap 버그수정 + 대표ID(max_area) 포함
# ==============================================================================
def merge_tracks(all_tracks,
                 iou_thresh    = MERGE_IOU_THRESH,
                 max_frame_gap = MERGE_MAX_FRAME_GAP,
                 conveyor_vx   = 0.0,
                 conveyor_vy   = 0.0,
                 partial_area_ratio = PARTIAL_AREA_RATIO,
                 edge_tolerance_ratio = EDGE_TOLERANCE_RATIO,
                 cross_axis_tolerance_ratio = CROSS_AXIS_TOLERANCE_RATIO,
                 min_track_frames_for_merge = MIN_TRACK_FRAMES_FOR_MERGE,
                 debug_log = None,
                 video_id  = None,
                 verbose_groups = False):
    if len(all_tracks) <= 1:
        return all_tracks

    track_profiles = build_spatial_order(all_tracks, conveyor_vx, conveyor_vy)

    def bbox_area(b):
        return (b[2]-b[0]) * (b[3]-b[1])

    def bbox_iou(a, b):
        ax1,ay1,ax2,ay2 = a
        bx1,by1,bx2,by2 = b
        ix1=max(ax1,bx1); iy1=max(ay1,by1)
        ix2=min(ax2,bx2); iy2=min(ay2,by2)
        inter = max(0,ix2-ix1)*max(0,iy2-iy1)
        if inter == 0: return 0.0
        ua = (ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter
        return inter/ua

    def center_dist(a, b):
        ax1,ay1,ax2,ay2 = a
        bx1,by1,bx2,by2 = b
        return np.sqrt(((ax1+ax2)/2-(bx1+bx2)/2)**2 +
                       ((ay1+ay2)/2-(by1+by2)/2)**2)

    def bbox_diag(b):
        x1,y1,x2,y2 = b
        return np.sqrt((x2-x1)**2+(y2-y1)**2)

    track_info = {}
    for tid, hist in all_tracks.items():
        if not hist:
            continue
        frames_ = [h[0] for h in hist]
        bboxes  = [h[1] for h in hist]
        areas   = [bbox_area(b) for b in bboxes]
        track_info[tid] = {
            'first_frame': min(frames_),
            'last_frame' : max(frames_),
            'first_bbox' : bboxes[0],
            'last_bbox'  : bboxes[-1],
            'max_area'   : max(areas),
            'frames'     : set(frames_),
            'n_frames'   : len(hist),
        }

    parent = {tid: tid for tid in track_info}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        parent[find(x)] = find(y)

    speed = np.sqrt(conveyor_vx**2 + conveyor_vy**2)
    is_horizontal = speed < 0.5 or abs(conveyor_vx) >= abs(conveyor_vy)

    merge_reasons = {}

    tids = list(track_info.keys())
    for i in range(len(tids)):
        for j in range(i+1, len(tids)):
            ti, tj = tids[i], tids[j]
            info_i, info_j = track_info[ti], track_info[tj]

            if info_i['n_frames'] < min_track_frames_for_merge or \
               info_j['n_frames'] < min_track_frames_for_merge:
                continue

            range_overlap = (info_i['first_frame'] < info_j['last_frame'] and
                              info_j['first_frame'] < info_i['last_frame'])
            if range_overlap:
                continue

            if info_i['last_frame'] <= info_j['first_frame']:
                earlier, later = info_i, info_j
                eid, lid = ti, tj
            elif info_j['last_frame'] <= info_i['first_frame']:
                earlier, later = info_j, info_i
                eid, lid = tj, ti
            else:
                continue

            dt = later['first_frame'] - earlier['last_frame']
            if dt > max_frame_gap:
                continue

            is_partial = later['max_area'] < earlier['max_area'] * partial_area_ratio

            if is_partial:
                ex1,ey1,ex2,ey2 = earlier['last_bbox']
                ex1 += conveyor_vx*dt; ex2 += conveyor_vx*dt
                ey1 += conveyor_vy*dt; ey2 += conveyor_vy*dt
                lx1,ly1,lx2,ly2 = later['first_bbox']

                if is_horizontal:
                    e_w = ex2 - ex1
                    edge_tolerance_px = e_w * edge_tolerance_ratio

                    edge_gap = min(abs(ex1-lx1), abs(ex1-lx2), abs(ex2-lx1), abs(ex2-lx2))
                    cross_overlap = min(ey2,ly2) - max(ey1,ly1)
                    cross_tol = -max(ey2-ey1, ly2-ly1) * cross_axis_tolerance_ratio
                    ok = edge_gap <= edge_tolerance_px and cross_overlap > cross_tol
                else:
                    e_h = ey2 - ey1
                    edge_tolerance_px = e_h * edge_tolerance_ratio

                    edge_gap = min(abs(ey1-ly1), abs(ey1-ly2), abs(ey2-ly1), abs(ey2-ly2))
                    cross_overlap = min(ex2,lx2) - max(ex1,lx1)
                    cross_tol = -max(ex2-ex1, lx2-lx1) * cross_axis_tolerance_ratio
                    ok = edge_gap <= edge_tolerance_px and cross_overlap > cross_tol

                if debug_log is not None:
                    debug_log.append({
                        'video_id'         : video_id,
                        'eid'              : eid,
                        'lid'              : lid,
                        'is_horizontal'    : is_horizontal,
                        'dt'               : dt,
                        'earlier_max_area' : earlier['max_area'],
                        'later_max_area'   : later['max_area'],
                        'area_ratio'       : later['max_area'] / max(earlier['max_area'], 1e-6),
                        'edge_gap'         : edge_gap,
                        'edge_tolerance_px': edge_tolerance_px,
                        'gap_over_tol'     : edge_gap / max(edge_tolerance_px, 1e-6),
                        'cross_overlap'    : cross_overlap,
                        'cross_tol'        : cross_tol,
                        'merged'           : ok,
                    })

                if ok:
                    union(eid, lid)
                    merge_reasons[(eid, lid)] = f"is_partial(dt={dt}, edge_gap={edge_gap:.1f}, tol={edge_tolerance_px:.1f})"
                continue

            prof_e = track_profiles.get(eid)
            prof_l = track_profiles.get(lid)
            if prof_e and prof_l:
                if not is_same_lane(prof_e, prof_l):
                    continue
                if not depth_order_consistent(prof_e, prof_l, conveyor_vy):
                    continue

            extrap = (earlier['last_bbox'][0]+conveyor_vx*dt, earlier['last_bbox'][1]+conveyor_vy*dt,
                     earlier['last_bbox'][2]+conveyor_vx*dt, earlier['last_bbox'][3]+conveyor_vy*dt)
            first_bbox = later['first_bbox']

            iou  = bbox_iou(extrap, first_bbox)
            dist = center_dist(extrap, first_bbox)
            diag = bbox_diag(earlier['last_bbox'])

            if iou >= iou_thresh or dist <= diag * 0.4:
                union(eid, lid)
                merge_reasons[(eid, lid)] = (f"normal(dt={dt}, iou={iou:.3f}, "
                                             f"dist={dist:.1f}, diag*0.4={diag*0.4:.1f}, "
                                             f"cx_e={prof_e['cx']:.0f}, cx_l={prof_l['cx']:.0f})") \
                                             if prof_e and prof_l else \
                                             f"normal(dt={dt}, iou={iou:.3f}, dist={dist:.1f}, diag*0.4={diag*0.4:.1f})"

    groups = defaultdict(list)
    for tid in tids:
        groups[find(tid)].append(tid)

    if verbose_groups:
        print(f"  [{video_id}] merge_tracks 결과 상세:")
        for root, members in groups.items():
            if len(members) > 1:
                print(f"    그룹(root={root}): {sorted(members)}")
                for (e, l), reason in merge_reasons.items():
                    if e in members and l in members:
                        print(f"      ID{e} <- ID{l}  |  {reason}")

    merged = {}
    for root, members in groups.items():
        combined = []
        for tid in members:
            combined.extend(all_tracks[tid])
        combined.sort(key=lambda x: x[0])

        rep_id = max(members, key=lambda m: track_info[m]['max_area'])
        merged[rep_id] = combined

    return merged


# ==============================================================================
# 2패스 추론 — ★ min_track_frames_for_count 최종 카운트 필터 포함
# ==============================================================================
def run_two_pass_tracking(video_path, precomputed_roi=None, verbose=True,
                          debug_log=None, verbose_groups=False,
                          min_track_frames_for_count=MIN_TRACK_FRAMES_FOR_COUNT):
    video_id_ = os.path.splitext(os.path.basename(video_path))[0]

    lv = os.path.join(WORK_DIR, os.path.basename(video_path))
    if not os.path.exists(lv):
        shutil.copy(video_path, lv)

    if precomputed_roi is not None:
        roi = precomputed_roi
    else:
        roi = get_motion_roi_bbox(lv)

    frames, fps = extract_frames(lv, LABEL_INTERVAL_SEC)

    KalmanBoxTracker.count = 0
    tracker1 = ByteTracker()
    all_dets  = []

    for fi, (frame, t) in enumerate(frames):
        if roi is not None:
            rx1, ry1, rx2, ry2 = map(int, roi)
            crop = frame[ry1:ry2, rx1:rx2].copy()
        else:
            crop = frame.copy()
        if crop.size == 0:
            all_dets.append([])
            continue
        dets = ort_detect(crop)
        all_dets.append(dets)
        tracker1.update(dets, fi)

    raw1   = tracker1.get_all_tracks()
    vx, vy = estimate_conveyor_speed(raw1)

    all_bboxes = [h[1] for hist in raw1.values() for h in hist]
    avg_w = float(np.mean([(b[2]-b[0]) for b in all_bboxes])) if all_bboxes else 150.0
    max_miss, merge_gap = compute_dynamic_params(vx, vy, LABEL_INTERVAL_SEC, avg_w)

    if verbose:
        print(f"  [1패스] raw tracks: {len(raw1)}")
        print(f"  속도 추정: vx={vx:.2f}  vy={vy:.2f} px/frame  (speed={np.sqrt(vx**2+vy**2):.2f})")
        print(f"  동적 파라미터: max_miss={max_miss}  merge_gap={merge_gap}  avg_bbox_w={avg_w:.1f}px")

    KalmanBoxTracker.count = 0
    tracker2 = ByteTracker(max_miss=max_miss, conveyor_vx=vx, conveyor_vy=vy)
    for fi, dets in enumerate(all_dets):
        tracker2.update(dets, fi)

    raw2   = tracker2.get_all_tracks()
    merged = merge_tracks(raw2, max_frame_gap=merge_gap, conveyor_vx=vx, conveyor_vy=vy,
                          debug_log=debug_log, video_id=video_id_, verbose_groups=verbose_groups)

    n_before_count_filter = len(merged)

    merged = {tid: hist for tid, hist in merged.items()
              if len(hist) >= min_track_frames_for_count}

    if verbose:
        n_filtered = n_before_count_filter - len(merged)
        extra = f"  (카운트 필터로 {n_filtered}개 제외)" if n_filtered > 0 else ""
        print(f"  [2패스] raw tracks: {len(raw2)}  →  merged: {n_before_count_filter}  →  최종: {len(merged)}{extra}")

    return merged, vx, vy, max_miss, merge_gap, all_dets, frames, roi


print('✅ 트래킹 파이프라인 준비 완료')

✅ 트래킹 파이프라인 준비 완료


## STAGE 3. Rail Segmentatoin & Scale

In [8]:
# ==============================================================================
# [4-1] 레일 검출 — YOLOv8-seg ONNX 후처리 + 룰베이스 보정
# ==============================================================================
from sklearn.linear_model import RANSACRegressor

rail_session = ort.InferenceSession(RAIL_ONNX_PATH, providers=providers)
print("RAIL:", [(i.name, i.shape) for i in rail_session.get_inputs()],
      rail_session.get_providers()[0])


def letterbox(img, new_shape=(640, 640), color=(114, 114, 114)):
    """원본 비율 유지하며 정사각형으로 패딩 리사이즈"""
    h, w = img.shape[:2]
    scale = min(new_shape[0]/h, new_shape[1]/w)
    nh, nw = int(round(h*scale)), int(round(w*scale))
    resized = cv2.resize(img, (nw, nh))

    canvas = np.full((new_shape[0], new_shape[1], 3), color, dtype=np.uint8)
    pad_top  = (new_shape[0] - nh) // 2
    pad_left = (new_shape[1] - nw) // 2
    canvas[pad_top:pad_top+nh, pad_left:pad_left+nw] = resized
    return canvas, scale, pad_left, pad_top


def preprocess_rail(img_bgr, imgsz=640):
    """이미지 → ONNX 입력 텐서"""
    canvas, scale, pad_left, pad_top = letterbox(img_bgr, (imgsz, imgsz))
    tensor = canvas[:, :, ::-1].astype(np.float32) / 255.0
    tensor = tensor.transpose(2, 0, 1)[np.newaxis]
    return tensor, scale, pad_left, pad_top


def xywh2xyxy(x):
    y = np.copy(x)
    y[..., 0] = x[..., 0] - x[..., 2] / 2
    y[..., 1] = x[..., 1] - x[..., 3] / 2
    y[..., 2] = x[..., 0] + x[..., 2] / 2
    y[..., 3] = x[..., 1] + x[..., 3] / 2
    return y


def nms_rail(boxes, scores, iou_thresh=0.45):
    idxs = scores.argsort()[::-1]
    keep = []
    while len(idxs) > 0:
        i = idxs[0]
        keep.append(i)
        if len(idxs) == 1:
            break
        rest = idxs[1:]

        xx1 = np.maximum(boxes[i,0], boxes[rest,0])
        yy1 = np.maximum(boxes[i,1], boxes[rest,1])
        xx2 = np.minimum(boxes[i,2], boxes[rest,2])
        yy2 = np.minimum(boxes[i,3], boxes[rest,3])

        inter = np.maximum(0, xx2-xx1) * np.maximum(0, yy2-yy1)
        area_i = (boxes[i,2]-boxes[i,0]) * (boxes[i,3]-boxes[i,1])
        area_r = (boxes[rest,2]-boxes[rest,0]) * (boxes[rest,3]-boxes[rest,1])
        iou = inter / (area_i + area_r - inter + 1e-9)

        idxs = rest[iou < iou_thresh]
    return keep


def process_mask(protos, mask_coeffs, boxes_xyxy_640, orig_h, orig_w,
                 scale, pad_left, pad_top, imgsz=640):
    """prototype × mask_coeffs → 원본 해상도 이진 마스크로 재구성"""
    c, mh, mw = protos.shape
    protos_flat = protos.reshape(c, -1)
    masks = mask_coeffs @ protos_flat
    masks = 1 / (1 + np.exp(-masks))
    masks = masks.reshape(-1, mh, mw)

    final_masks = []
    for i in range(masks.shape[0]):
        m = cv2.resize(masks[i], (imgsz, imgsz), interpolation=cv2.INTER_LINEAR)

        nh = int(round(orig_h * scale))
        nw = int(round(orig_w * scale))
        m = m[pad_top:pad_top+nh, pad_left:pad_left+nw]
        m = cv2.resize(m, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)

        x1, y1, x2, y2 = boxes_xyxy_640[i]
        x1 = (x1 - pad_left) / scale; y1 = (y1 - pad_top) / scale
        x2 = (x2 - pad_left) / scale; y2 = (y2 - pad_top) / scale
        x1, y1 = max(0,int(x1)), max(0,int(y1))
        x2, y2 = min(orig_w,int(x2)), min(orig_h,int(y2))

        mask_bin = np.zeros((orig_h, orig_w), dtype=np.uint8)
        mask_bin[y1:y2, x1:x2] = (m[y1:y2, x1:x2] > 0.5).astype(np.uint8)
        final_masks.append(mask_bin)

    return final_masks


def run_yolov8_seg_onnx(session, img_bgr, conf_thresh=0.25, iou_thresh=0.45, imgsz=640):
    """
    YOLOv8-seg ONNX 전체 추론
    반환: [{'box':(x1,y1,x2,y2), 'conf':float, 'mask':(H,W) uint8}, ...]
    """
    orig_h, orig_w = img_bgr.shape[:2]
    tensor, scale, pad_left, pad_top = preprocess_rail(img_bgr, imgsz)

    input_name = session.get_inputs()[0].name
    outputs = session.run(None, {input_name: tensor})

    pred = outputs[0][0]      # (4+1+nc+32, N)
    protos = outputs[1][0]    # (32, mh, mw)
    pred = pred.T              # (N, 4+1+nc+32)

    nc = 1   # 클래스 개수 (rail 하나)
    boxes_xywh = pred[:, :4]
    class_scores = pred[:, 4:4+nc]
    mask_coeffs = pred[:, 4+nc:]

    conf = class_scores[:, 0]
    mask_valid = conf > conf_thresh
    if not mask_valid.any():
        return []

    boxes_xywh = boxes_xywh[mask_valid]
    conf = conf[mask_valid]
    mask_coeffs = mask_coeffs[mask_valid]

    boxes_xyxy = xywh2xyxy(boxes_xywh)

    keep = nms_rail(boxes_xyxy, conf, iou_thresh)
    boxes_xyxy = boxes_xyxy[keep]
    conf = conf[keep]
    mask_coeffs = mask_coeffs[keep]

    masks = process_mask(protos, mask_coeffs, boxes_xyxy,
                         orig_h, orig_w, scale, pad_left, pad_top, imgsz)

    results = []
    for i in range(len(boxes_xyxy)):
        x1, y1, x2, y2 = boxes_xyxy[i]
        x1 = (x1 - pad_left) / scale; y1 = (y1 - pad_top) / scale
        x2 = (x2 - pad_left) / scale; y2 = (y2 - pad_top) / scale
        results.append({
            'box': (int(x1), int(y1), int(x2), int(y2)),
            'conf': float(conf[i]),
            'mask': masks[i],
        })
    return results


def refine_rail_from_mask(mask, W, H,
                          center_tol=0.35,
                          ransac_residual=15,
                          ratio_min=0.15,
                          ratio_max=0.85):
    """
    이진 마스크 → 규칙 기반으로 정리된 사다리꼴 4점 반환
    규칙: 1)중심 위치  2)near를 이미지 맨 아래로 강제  3)far<near + 비율 범위
    """
    mask_bin = (mask > 0).astype(np.uint8)

    ys_all, xs_all = np.where(mask_bin > 0)
    if len(xs_all) == 0:
        return None, "마스크 비어있음"

    # 규칙 1: 중심 위치
    cx = xs_all.mean()
    if abs(cx - W/2) / W > center_tol:
        return None, f"중심 이탈 (cx={cx:.0f}, center={W/2:.0f})"

    left_pts, right_pts = [], []
    y_min, y_max = ys_all.min(), ys_all.max()
    for y in range(y_min, y_max+1):
        xs = np.where(mask_bin[y] > 0)[0]
        if len(xs) < 3:
            continue
        left_pts.append([xs.min(), y])
        right_pts.append([xs.max(), y])

    if len(left_pts) < 10 or len(right_pts) < 10:
        return None, "경계 점 부족"

    def fit_line(pts):
        pts = np.array(pts)
        X = pts[:,1].reshape(-1,1)
        y = pts[:,0]
        ransac = RANSACRegressor(residual_threshold=ransac_residual, min_samples=10)
        ransac.fit(X, y)
        return ransac

    left_model  = fit_line(left_pts)
    right_model = fit_line(right_pts)

    # 규칙 2: near y를 이미지 맨 아래로 강제
    near_y = H - 1
    far_y  = y_min

    x_near_left  = float(left_model.predict([[near_y]])[0])
    x_near_right = float(right_model.predict([[near_y]])[0])
    x_far_left   = float(left_model.predict([[far_y]])[0])
    x_far_right  = float(right_model.predict([[far_y]])[0])

    near_w = abs(x_near_right - x_near_left)
    far_w  = abs(x_far_right  - x_far_left)

    # 규칙 3-a: 사다리꼴 형태 (far < near)
    if far_w >= near_w:
        return None, f"사다리꼴 형태 위반 (near={near_w:.0f} far={far_w:.0f})"

    # 규칙 3-b: 비율 범위
    ratio = far_w / (near_w + 1e-6)
    if not (ratio_min < ratio < ratio_max):
        return None, f"원근 비율 이상 (ratio={ratio:.2f})"

    clean_quad = {
        'near_left':  (x_near_left,  near_y),
        'near_right': (x_near_right, near_y),
        'far_left':   (x_far_left,   far_y),
        'far_right':  (x_far_right,  far_y),
        'near_width_px': near_w,
        'far_width_px':  far_w,
    }
    return clean_quad, "OK"


def run_rail_inference_refined_onnx(session, img_bgr, conf_thresh=0.25):
    """ONNX 추론 + 마스크재구성 + 룰베이스 후처리까지 한 번에"""
    H, W = img_bgr.shape[:2]
    dets = run_yolov8_seg_onnx(session, img_bgr, conf_thresh=conf_thresh)

    if not dets:
        return None, "탐지 실패"

    best = max(dets, key=lambda d: d['conf'])
    clean_quad, reason = refine_rail_from_mask(best['mask'], W, H)
    return clean_quad, reason


# ──────────────────────────────────────────────────────────────────────────────
# 레일 quad 확정 (영상 단위): 균등 30프레임 → 성공 quad들의 중앙값 합성
# ★ f̂ 통합판: 소실점 계산을 위해 4점 x좌표까지 수집
# ──────────────────────────────────────────────────────────────────────────────
RAIL_N_SAMPLE = 30
RAIL_MIN_OK   = 3

def estimate_rail_quad(frames, session=None, conf_thresh=0.25):
    """반환: near/far 폭 + 4점 x좌표 + 프레임 크기, 실패 시 None"""
    if session is None:
        session = rail_session
    if not frames:
        return None
    H, W = frames[0][0].shape[:2]
    n = len(frames)
    idxs = np.unique(np.linspace(0, n-1, min(RAIL_N_SAMPLE, n)).astype(int))

    cols = {k: [] for k in ['nl', 'nr', 'fl', 'fr', 'fy']}
    for i in idxs:
        quad, reason = run_rail_inference_refined_onnx(session, frames[i][0], conf_thresh)
        if quad is None:
            continue
        cols['nl'].append(quad['near_left'][0]);  cols['nr'].append(quad['near_right'][0])
        cols['fl'].append(quad['far_left'][0]);   cols['fr'].append(quad['far_right'][0])
        cols['fy'].append(quad['far_left'][1])

    if len(cols['nl']) < RAIL_MIN_OK:
        return None
    q = {k: float(np.median(v)) for k, v in cols.items()}
    q['near_y'] = float(H - 1); q['W'] = W; q['H'] = H
    q['near_w'] = q['nr'] - q['nl']; q['far_w'] = q['fr'] - q['fl']
    q['far_y']  = q['fy']
    q['n_ok']   = len(cols['nl'])
    return q


def rail_px_width_at_y(quad, y):
    t = (quad['near_y'] - y) / max(quad['near_y'] - quad['far_y'], 1e-6)
    t = float(np.clip(t, 0.0, 1.15))
    w = quad['near_w'] + (quad['far_w'] - quad['near_w']) * t
    return max(w, 1.0)


def rail_scale_at_y(quad, y):
    return RAIL_WIDTH_CM / rail_px_width_at_y(quad, y)


# ── f̂ (focal 추정) 기하 유틸 ──────────────────────────────────────────────────
def vanish_y_of(q):
    """레일 양변 연장 교점 y — 직선교차/폭외삽 두 방식 평균 (교차검증 겸)"""
    ny, fy = q['near_y'], q['fy']
    denom = (fy - ny)
    aL = (q['fl'] - q['nl']) / denom
    aR = (q['fr'] - q['nr']) / denom
    y_lines = ny + (q['nr'] - q['nl']) / (aL - aR) if abs(aL - aR) > 1e-9 else None
    y_width = ny - q['near_w'] * (ny - fy) / (q['near_w'] - q['far_w'] + 1e-9)
    if y_lines is None:
        return float(y_width)
    return float((y_lines + y_width) / 2)


def quad_feats(q):
    """focal 회귀 입력 피처 4개 (전부 프레임 크기로 정규화 → 해상도 불변)"""
    y_v = vanish_y_of(q)
    cy = q['H'] / 2
    return [(cy - y_v) / q['H'],
            q['far_w'] / q['near_w'],
            q['near_w'] / q['W'],
            (q['near_y'] - q['fy']) / q['H']]

print("✅ 레일 파이프라인 준비 (f̂ 통합판)")

RAIL: [('images', [1, 3, 640, 640])] CUDAExecutionProvider
✅ 레일 파이프라인 준비 (f̂ 통합판)


## STAGE 4. Depth Estimation

In [9]:
# ==============================================================================
# [5-1] DA3 depth (원본 클래스 보존 + raw 출력 추가)
# ==============================================================================
_providers = providers

class DA3Depth:
    def __init__(self, path):
        self.sess=ort.InferenceSession(path, providers=_providers)
        self.iname=self.sess.get_inputs()[0].name
        self.oname=self.sess.get_outputs()[0].name
        sh=self.sess.get_inputs()[0].shape
        self.fh=sh[2] if isinstance(sh[2],int) else None
        self.fw=sh[3] if isinstance(sh[3],int) else None
        print("DA3:", sh, self.sess.get_providers()[0])

    def predict_raw(self, bgr):
        """focal 무관 raw depth (원본 해상도로 리사이즈)"""
        oh,ow=bgr.shape[:2]
        if self.fh and self.fw: th,tw=self.fh,self.fw
        else:
            P=14; th=max((oh//P)*P,P); tw=max((ow//P)*P,P)
        rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)
        r=cv2.resize(rgb,(tw,th)).astype(np.float32)/255.0
        t=r.transpose(2,0,1)[None]
        raw=self.sess.run([self.oname],{self.iname:t})[0][0,0]
        return cv2.resize(raw,(ow,oh))

    def predict_metric(self, bgr, focal_mm, sensor_w_mm):
        oh,ow=bgr.shape[:2]
        focal_px=focal_mm/sensor_w_mm*ow
        raw=self.predict_raw(bgr)
        return focal_px*raw/300.0, focal_px

def focal_px_of(focal_mm, frame_w):
    return focal_mm / SENSOR_W * frame_w

da3_model = DA3Depth(DA3_PATH)
print("✅ da3_model 준비")

DA3: [1, 3, 280, 504] CUDAExecutionProvider
✅ da3_model 준비


## STAGE 5. 영상 레코드 빌드 (Cache)

In [10]:
# ==============================================================================
# [6-1] 수정판 — tid 포함, 캐시를 Google Drive에 저장
# ==============================================================================
RECORDS_PATH_V2 = os.path.join(PROJECT_DIR, 'video_records_0713_with_yolov6.pkl')  # ★ 파일명 변경

def build_video_record_with_tid(video_path):
    vid = os.path.splitext(os.path.basename(video_path))[0]
    lv = os.path.join(WORK_DIR, os.path.basename(video_path))   # 트래킹용 로컬 임시복사는 그대로 유지
    if not os.path.exists(lv):
        shutil.copy(video_path, lv)
    roi_pre = get_roi_for_train_video(lv)
    merged, vx, vy, max_miss, merge_gap, all_dets, frames, roi = \
        run_two_pass_tracking(video_path, precomputed_roi=roi_pre, verbose=False)
    quad = estimate_rail_quad(frames)
    H, W = frames[0][0].shape[:2] if frames else (0, 0)
    rx1, ry1, rx2, ry2 = map(int, roi) if roi is not None else (0, 0, W, H)

    reps = []
    for tid, hist in merged.items():
        best = max(hist, key=lambda h: (h[1][2]-h[1][0]) * (h[1][3]-h[1][1]))
        fi, (x1, y1, x2, y2), conf = best
        ox1, oy1 = int(x1 + rx1), int(y1 + ry1)
        ox2, oy2 = int(x2 + rx1), int(y2 + ry1)
        ox1, oy1 = max(0, ox1), max(0, oy1)
        ox2, oy2 = min(W, ox2), min(H, oy2)
        if ox2 <= ox1 + 3 or oy2 <= oy1 + 3:
            continue
        reps.append((tid, (ox2-ox1)*(oy2-oy1), fi, (ox1, oy1, ox2, oy2)))

    raw_cache = {}
    tracks = []
    for tid, area, fi, (ox1, oy1, ox2, oy2) in reps:
        frame = frames[fi][0]
        if fi not in raw_cache:
            raw_cache[fi] = da3_model.predict_raw(frame)
        raw = raw_cache[fi]
        rc = frame[oy1:oy2, ox1:ox2]
        dc = raw[oy1:oy2, ox1:ox2]
        if rc.size == 0:
            continue
        rc_rgb = cv2.cvtColor(rc, cv2.COLOR_BGR2RGB)          # ★ BGR→RGB 변환

        tracks.append({
            'tid':      tid,
            'rgb':      cv2.resize(rc_rgb, (CROP_SIZE, CROP_SIZE)).astype(np.uint8),  # ★ rc → rc_rgb (변환 결과를 실제로 저장)
            'raw':      cv2.resize(dc, (CROP_SIZE, CROP_SIZE)).astype(np.float32),
            'raw_med':  float(np.median(dc)),
            'bbox_w':   float(ox2-ox1),
            'bbox_h':   float(oy2-oy1),
            'y_bottom': float(oy2),
            'area':     float(area),
        })

    return {'vid': vid, 'W': W, 'H': H, 'quad': quad,
            'vx': vx, 'vy': vy, 'tracks': tracks}


if os.path.exists(RECORDS_PATH_V2):
    with open(RECORDS_PATH_V2, 'rb') as f:
        video_records_tid = pickle.load(f)
    print(f"캐시 로드(Drive): {len(video_records_tid)}개")
else:
    video_records_tid = {}
    for vp in tqdm(train_videos, desc="영상 레코드(tid포함)"):
        vid = os.path.splitext(os.path.basename(vp))[0]
        try:
            video_records_tid[vid] = build_video_record_with_tid(vp)
        except Exception as e:
            print(f"  ❌ {vid}: {e}")
    with open(RECORDS_PATH_V2, 'wb') as f:
        pickle.dump(video_records_tid, f)
    print(f"캐시 저장(Drive): {RECORDS_PATH_V2}")

# ── RGB 저장 검증: 갈색 박스는 R채널 평균 > B채널 평균이어야 함 ──────────────
_all = np.stack([t['rgb'] for r in video_records_tid.values() for t in r['tracks']])
_r, _b = _all[..., 0].mean(), _all[..., 2].mean()
print(f"채널 평균  ch0={_r:.1f}  ch2={_b:.1f}  →  {'✅ RGB로 저장됨' if _r > _b else '❌ BGR로 저장됨! 캐시 pkl 삭제 후 이 셀 재실행 필요'}")


캐시 로드(Drive): 100개
채널 평균  ch0=136.3  ch2=117.5  →  ✅ RGB로 저장됨


## STAGE 6. 학습 DATASET 만들기

In [14]:
# ==============================================================================
# f̂(focal 추정) 회귀 학습
# ==============================================================================
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_predict, KFold

fv_vids, fv_X, fv_y = [], [], []
for vid, rec in video_records_tid.items():
    if vid in label_map and rec['quad'] is not None:
        fv_vids.append(vid)
        fv_X.append(quad_feats(rec['quad']))
        fv_y.append(label_map[vid]['camera']['focal_length_mm'])
fv_X, fv_y = np.array(fv_X), np.array(fv_y)

cv = KFold(5, shuffle=True, random_state=42)
f_oof = cross_val_predict(LinearRegression(), fv_X, fv_y, cv=cv)
print(f"f̂ CV: MAE {np.abs(f_oof - fv_y).mean():.3f}mm  corr {np.corrcoef(f_oof, fv_y)[0,1]:.3f}  "
      f"(베이스라인 {np.abs(fv_y - fv_y.mean()).mean():.3f}mm)")

f_reg = LinearRegression().fit(fv_X, fv_y)
F_COEF, F_INT = f_reg.coef_.tolist(), float(f_reg.intercept_)
f_hat_map = {v: float(f) for v, f in zip(fv_vids, f_oof)}


def fhat_from_quad(quad):
    if quad is None:
        return FOCAL_MEAN
    return float(np.dot(F_COEF, quad_feats(quad)) + F_INT)

print(f"f_hat_map 준비 완료: {len(f_hat_map)}개 영상")

def scalars_rail(tr, quad):
    s = rail_scale_at_y(quad, tr['y_bottom'])
    return np.array([tr['bbox_w'] * s, tr['bbox_h'] * s], np.float32)

def scalars_da3(tr):
    g = tr['raw_med'] / 300.0
    return np.array([tr['bbox_w'] * g, tr['bbox_h'] * g], np.float32)

# SC_FALLBACK_K: rail scale이 없는 track에 대한 da3 기반 스칼라 보정계수
#   — build_gt_dataset_branch1_v2()/save_dataset()의 fallback_k와 동일한 방식으로 재계산
_ratios_for_fallback = []
for vid, rec in video_records_tid.items():
    if vid not in label_map or rec['quad'] is None:
        continue
    for tr in rec['tracks']:
        sc_r = scalars_rail(tr, rec['quad'])
        sc_d = scalars_da3(tr)
        _ratios_for_fallback.append(sc_r / np.maximum(sc_d, 1e-6))

SC_FALLBACK_K = float(np.median(np.concatenate(_ratios_for_fallback))) if _ratios_for_fallback else 1.0
print(f"SC_FALLBACK_K = {SC_FALLBACK_K:.4f}")

f̂ CV: MAE 0.698mm  corr 0.979  (베이스라인 3.814mm)
f_hat_map 준비 완료: 100개 영상
SC_FALLBACK_K = 52.2905


In [17]:
# ==============================================================================
# 매핑 엑셀 로드 → mapping_df / usable_df 생성
# ==============================================================================
import pandas as pd

_raw_df = pd.read_excel(MAPPING_XLSX_PATH, sheet_name='mapping')

mapping_df = _raw_df[['video_id', 'box_id_json', 'w_cm', 'd_cm', 'h_cm',
                       'volume_cm3', 'VERSION 6', 'note',
                       'focal_length_mm', 'sensor_width_mm', 'sensor_height_mm']].copy()
mapping_df = mapping_df.rename(columns={'VERSION 6': 'new_tracked_unique_id'})

mapping_df['is_removed_note'] = (
    mapping_df['note'].astype(str).str.strip().str.upper() == 'REMOVED'
)

def _is_usable(v):
    if isinstance(v, str):
        return v.strip().upper() != 'NONE'
    return True

mapping_df['_usable'] = mapping_df['new_tracked_unique_id'].apply(_is_usable)

usable_df   = mapping_df[mapping_df['_usable']].copy()
excluded_df = mapping_df[~mapping_df['_usable']].copy()
usable_df['new_tracked_unique_id'] = usable_df['new_tracked_unique_id'].astype(int)

print(f"매핑 엑셀: 총 {len(mapping_df)}행")
print(f"  학습 가능(usable_df) {len(usable_df)}행")
print(f"  제외(excluded_df) {len(excluded_df)}행")
print(f"  note=REMOVED 표시된 행: {mapping_df['is_removed_note'].sum()}행")

매핑 엑셀: 총 918행
  학습 가능(usable_df) 917행
  제외(excluded_df) 1행
  note=REMOVED 표시된 행: 12행


In [ ]:
# ==============================================================================
# CELL 20 (최종본, 자체완결형): TRAIN 100개 영상 — 채점 + CNN 학습 데이터셋 생성
#   ★ 프레임 선택 로직 (4단계):
#     0) track 내 area가 가장 큰 프레임은 clipped 여부와 무관하게 무조건 포함
#     1) track 내 area 상위 TOP_AREA_RATIO_KEEP(%) 까지만 후보로 인정
#        (절대 임계값 대신 상대 순위 사용 → track마다 분포가 달라도 공평하게 적용)
#     2) 후보가 MIN_FRAMES_PER_TRACK보다 적으면 완화해서 최소 개수 보장
#     3) (clipped 여부, -area) 우선순위로 정렬 + 최소 시간 간격 유지하며
#        MAX_FRAMES_PER_TRACK개까지 최종 선택 (부족하면 간격 무시하고 최소치 채움)
#   ★ crop 색상: BGR → RGB 변환 유지
#   ★ crop 리사이즈: 없음 — 원본 비율 그대로 저장 (옵션 A, object array)
#   ★ mapping_df / usable_df는 이전 셀("매핑 엑셀 로드")에서 이미 생성된 것을 재사용
# ==============================================================================
import pandas as pd

# ── 헬퍼 함수 정의 ──
GLOBAL_MOTION_THRESH = MOTION_THRESHOLD
GLOBAL_RATIO_THRESH  = MOTION_RATIO_THRESH

def get_test_video_id(video_path):
    return os.path.splitext(os.path.basename(video_path))[0]


def get_roi_for_test_video(video_path,
                           motion_threshold=GLOBAL_MOTION_THRESH,
                           motion_ratio_thresh=GLOBAL_RATIO_THRESH):
    cap     = cv2.VideoCapture(video_path)
    fps     = cap.get(cv2.CAP_PROP_FPS)
    total_f = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    orig_w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    min_sample_frames = 20
    frame_step = max(1, min(int(fps * 0.5), max(total_f // min_sample_frames, 1)))
    extra = set(range(min(5, total_f))) | set(range(max(0, total_f - 5), total_f))
    sample_indices = np.array(
        sorted(set(np.arange(0, total_f, frame_step, dtype=int).tolist()) | extra)
    )

    sampled_frames = []
    for idx in sample_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if ret:
            sampled_frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
    cap.release()

    if len(sampled_frames) < 3:
        return None

    stack     = np.stack(sampled_frames, axis=0).astype(np.float32)
    bg_median = np.median(stack, axis=0)
    motion_count = np.zeros((orig_h, orig_w), dtype=np.float32)
    for frame_gray in sampled_frames:
        diff = np.abs(frame_gray.astype(np.float32) - bg_median)
        motion_count += (diff > motion_threshold).astype(np.float32)
    motion_ratio = motion_count / len(sampled_frames)

    duration_sec    = total_f / max(fps, 1)
    adaptive_thresh = motion_ratio_thresh
    if duration_sec < 5:
        adaptive_thresh = motion_ratio_thresh * 0.4
    elif duration_sec < 10:
        adaptive_thresh = motion_ratio_thresh * 0.7

    motion_mask = (motion_ratio >= adaptive_thresh).astype(np.uint8) * 255
    kernel      = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    motion_mask = cv2.morphologyEx(motion_mask, cv2.MORPH_CLOSE, kernel)
    motion_mask = cv2.morphologyEx(motion_mask, cv2.MORPH_OPEN,  kernel)

    contours, _ = cv2.findContours(motion_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return (0, 0, orig_w, orig_h)

    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    bx, by, bw, bh = cv2.boundingRect(contours[0])
    pad = 20
    return (max(0, bx-pad), max(0, by-pad),
            min(orig_w, bx+bw+pad), min(orig_h, by+bh+pad))


def get_roi_for_train_video(video_path,
                            motion_threshold=GLOBAL_MOTION_THRESH,
                            motion_ratio_thresh=GLOBAL_RATIO_THRESH):
    """test용 get_roi_for_test_video와 완전히 동일한 로직 (train 경로 대응 wrapper)"""
    return get_roi_for_test_video(video_path, motion_threshold, motion_ratio_thresh)


def score_test_predictions(pred_df, gt_dict):
    df = pred_df.copy()
    df['gt_count'] = df['video_id'].map(gt_dict)
    missing = df[df['gt_count'].isna()]
    if not missing.empty:
        print(f"⚠️ 정답이 없는 영상 {len(missing)}개 (채점 제외): {missing['video_id'].tolist()}")
        df = df.dropna(subset=['gt_count'])
    df['gt_count'] = df['gt_count'].astype(int)
    df['diff'] = df['pred_count'] - df['gt_count']
    df['exact_match'] = df['diff'] == 0
    df = df.sort_values('video_id').reset_index(drop=True)
    return df


def print_test_comparison_table(df):
    print(f"{'영상':<12}{'정답':>6}{'예측':>6}{'차이':>7}")
    print("-" * 32)
    for _, row in df.iterrows():
        diff = int(row['diff'])
        mark = "✅" if diff == 0 else ("🔺" if diff > 0 else "🔻")
        print(f"{row['video_id']:<12}{int(row['gt_count']):>4} {int(row['pred_count']):>5} {diff:>+5}  {mark}")
    n_total = len(df)
    n_exact = df['exact_match'].sum()
    mae     = df['diff'].abs().mean()
    print("\n" + "="*60)
    print(f"전체 평가 영상 수         : {n_total}")
    print(f"개수 정확히 일치         : {n_exact}개 ({n_exact/n_total*100:.1f}%)")
    print(f"개수 불일치 (과다검출)    : {(df['diff'] > 0).sum()}개")
    print(f"개수 불일치 (과소검출)    : {(df['diff'] < 0).sum()}개")
    print(f"평균 절대 오차(MAE, 개수) : {mae:.3f}")
    print("="*60)
    if n_total - n_exact > 0:
        print("\n틀린 영상 목록:")
        print(df[~df['exact_match']][['video_id', 'gt_count', 'pred_count', 'diff']].to_string(index=False))


def is_bbox_clipped(bbox, crop_w, crop_h, margin=5):
    """bbox가 crop(ROI) 경계에서 margin px 이내에 닿아있으면 잘린 것으로 간주"""
    x1, y1, x2, y2 = bbox[:4]
    return (x1 <= margin or y1 <= margin
            or x2 >= crop_w - margin or y2 >= crop_h - margin)


def select_frames_for_track(hist, crop_w, crop_h,
                            top_area_ratio_keep, min_frames, max_frames,
                            min_frame_gap, clip_margin):
    """
    track의 전체 hist에서 최종 학습용 프레임을 선택하는 함수.
    0) area가 가장 큰 프레임은 clipped 여부와 무관하게 무조건 최종 선택에 포함
    1) area 상위 top_area_ratio_keep(%)까지만 후보로 인정 (상대 순위 기준)
    2) 후보 수가 min_frames보다 적으면 완화해서 min_frames까지 채움
    3) (clipped, -area) 우선순위로 정렬 + 최소 시간 간격 유지하며
       max_frames개까지 선택 (0번에서 이미 선택된 max-area 프레임은 건너뛰고
       나머지 슬롯만 채움)
    4) 그래도 min_frames 미달이면 시간 간격 무시하고 채움
    반환: [(h, area), ...] 시간순 정렬된 리스트, max_area
    """
    areas = [(h[1][2]-h[1][0]) * (h[1][3]-h[1][1]) for h in hist]
    max_area = max(areas) if areas else 0
    if max_area <= 0:
        return [], max_area

    max_idx = int(np.argmax(areas))
    forced_item = (hist[max_idx], areas[max_idx])
    forced_fi   = hist[max_idx][0]

    n_total = len(hist)
    n_top = max(1, int(np.ceil(n_total * top_area_ratio_keep)))

    all_pairs_by_area = sorted(zip(hist, areas), key=lambda x: x[1], reverse=True)
    top_candidates = all_pairs_by_area[:n_top]

    if len(top_candidates) < min_frames:
        top_candidates = all_pairs_by_area[:min_frames]

    if not top_candidates:
        return [], max_area

    def _sort_key(item, cw=crop_w, ch=crop_h, margin=clip_margin):
        h, a = item
        bbox = h[1]
        clipped = is_bbox_clipped(bbox, cw, ch, margin=margin)
        return (clipped, -a)

    sorted_by_priority = sorted(top_candidates, key=_sort_key)

    selected = [forced_item]
    selected_frame_indices = [forced_fi]

    for h, a in sorted_by_priority:
        fi = h[0]
        if fi == forced_fi:
            continue
        if all(abs(fi - sfi) >= min_frame_gap for sfi in selected_frame_indices):
            selected.append((h, a))
            selected_frame_indices.append(fi)
        if len(selected) >= max_frames:
            break

    if len(selected) < min_frames:
        remaining_needed = min_frames - len(selected)
        already_fis = set(s[0][0] for s in selected)
        extra = [item for item in sorted_by_priority if item[0][0] not in already_fis]
        selected.extend(extra[:remaining_needed])

    selected = sorted(selected, key=lambda x: x[0][0])
    return selected, max_area


# ── 본 처리 시작 ─────────────────────────────────────────────────────────
# ★ MAPPING_XLSX_PATH 로드 + mapping_df/usable_df 생성 부분은 삭제됨
#   → 이전 셀("매핑 엑셀 로드 → mapping_df / usable_df 생성")에서 만든
#     usable_df를 그대로 재사용합니다.

with open(TRAIN_LABEL_PATH) as f:
    train_label_data = json.load(f)
train_gt_count = {
    v['video_id']: len(v['objects']) for v in train_label_data['videos']
}
print(f"Train 정답(GT) 로드 완료: {len(train_gt_count)}개 영상")

train_video_files = sorted(glob.glob(os.path.join(TRAIN_VIDEO_DIR, '**', '*.mp4'), recursive=True))
print(f"Train 영상 개수: {len(train_video_files)}")

MAX_FRAMES_PER_TRACK = 8
MIN_FRAMES_PER_TRACK = 2      # track당 최소 보장 프레임 수
TOP_AREA_RATIO_KEEP  = 0.5
MIN_FRAME_GAP_SEC    = 0.2
CLIP_MARGIN_PX       = 5

dataset_records   = []
train_results     = []
train_errors      = []
mismatch_report   = []

print(f"\nTrain 영상 {len(train_video_files)}개 — 트래킹 1회 실행하며 채점 + 데이터셋 동시 생성...")
t0 = time.time()

for i, vp in enumerate(train_video_files):
    video_id = get_test_video_id(vp)
    try:
        roi = get_roi_for_train_video(vp)
        if roi is None:
            print(f"  [{i+1}/{len(train_video_files)}] {video_id}  ROI 없음 → skip")
            continue

        merged, vx, vy, max_miss, merge_gap, all_dets, frames, used_roi = \
            run_two_pass_tracking(vp, precomputed_roi=roi, verbose=False)

        train_results.append({'video_id': video_id, 'pred_count': len(merged)})

        rows = usable_df[usable_df['video_id'] == video_id]
        if len(rows) == 0:
            continue

        rx1, ry1, rx2, ry2 = map(int, used_roi)
        crop_w, crop_h = rx2 - rx1, ry2 - ry1
        H, W = frames[0][0].shape[:2] if frames else (0, 0)

        expected_ids = set(rows['new_tracked_unique_id'].astype(int))
        actual_ids   = set(merged.keys())
        missing_ids  = expected_ids - actual_ids
        if missing_ids:
            mismatch_report.append({
                'video_id': video_id,
                'missing_ids': sorted(missing_ids),
                'expected': sorted(expected_ids),
                'actual': sorted(actual_ids),
            })

        for _, row in rows.iterrows():
            tracked_id = row['new_tracked_unique_id']
            if tracked_id not in merged:
                continue

            hist = merged[tracked_id]

            min_frame_gap = max(1, int(round(MIN_FRAME_GAP_SEC / LABEL_INTERVAL_SEC)))
            selected, max_area = select_frames_for_track(
                hist, crop_w, crop_h,
                top_area_ratio_keep=TOP_AREA_RATIO_KEEP,
                min_frames=MIN_FRAMES_PER_TRACK,
                max_frames=MAX_FRAMES_PER_TRACK,
                min_frame_gap=min_frame_gap,
                clip_margin=CLIP_MARGIN_PX,
            )
            if not selected or max_area <= 0:
                continue

            areas_all = [(h[1][2]-h[1][0]) * (h[1][3]-h[1][1]) for h in hist]
            frame_idx_of_max_area = hist[int(np.argmax(areas_all))][0]

            w_cm, h_cm, d_cm = row['w_cm'], row['h_cm'], row['d_cm']
            r_hw = h_cm / w_cm
            r_dw = d_cm / w_cm
            n_selected_for_track = len(selected)

            for (fi, (bx1, by1, bx2, by2), conf), area in selected:
                ox1, oy1 = int(bx1 + rx1), int(by1 + ry1)
                ox2, oy2 = int(bx2 + rx1), int(by2 + ry1)
                ox1, oy1 = max(0, ox1), max(0, oy1)
                ox2, oy2 = min(W, ox2), min(H, oy2)
                if ox2 <= ox1 + 3 or oy2 <= oy1 + 3:
                    continue

                clipped_flag = is_bbox_clipped((bx1, by1, bx2, by2), crop_w, crop_h,
                                               margin=CLIP_MARGIN_PX)

                frame_img = frames[fi][0]
                crop_bgr = frame_img[oy1:oy2, ox1:ox2]
                crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)

                dataset_records.append({
                    'video_id'   : video_id,
                    'tracked_id' : int(tracked_id),
                    'box_id_json': int(row['box_id_json']),
                    'rgb'        : crop_rgb,
                    'w_cm': w_cm, 'h_cm': h_cm, 'd_cm': d_cm,
                    'r_hw': r_hw, 'r_dw': r_dw,
                    'bbox_w_px'  : float(ox2-ox1),
                    'bbox_h_px'  : float(oy2-oy1),
                    'area_ratio' : float(area / max_area),
                    'conf'       : float(conf),
                    'is_clipped' : bool(clipped_flag),
                    'is_max_area': bool(fi == frame_idx_of_max_area),
                    'n_frames_in_track': n_selected_for_track,
                    'focal_length_mm' : row['focal_length_mm'],
                    'sensor_width_mm' : row['sensor_width_mm'],
                    'sensor_height_mm': row['sensor_height_mm'],
                    'frame_idx'  : fi,
                })

    except Exception as e:
        print(f"  ❌ [{video_id}] 에러: {e}")
        train_errors.append(video_id)
        continue

    if (i+1) % 10 == 0 or (i+1) == len(train_video_files):
        elapsed = time.time() - t0
        print(f"  [{i+1}/{len(train_video_files)}] 진행 중... 경과 {elapsed:.1f}s  누적샘플 {len(dataset_records)}개")

print(f"\n완료. 총 소요 시간: {time.time()-t0:.1f}s")
print(f"총 데이터셋 샘플: {len(dataset_records)}개")
if train_errors:
    print(f"⚠️ 에러난 영상: {train_errors}")

print(f"\n트랙 불일치 영상 수: {len(mismatch_report)}개")
for r in mismatch_report:
    print(f"  {r['video_id']}: 기대={r['expected']}  실제={r['actual']}  누락={r['missing_ids']}")


# ==============================================================================
# TRAIN 채점
# ==============================================================================
train_df = pd.DataFrame(train_results)
scored_train_df = score_test_predictions(train_df, train_gt_count)
print_test_comparison_table(scored_train_df)


# ==============================================================================
# 데이터셋 배열화 + 저장 (★ 옵션 A: rgb는 object array, resize 없음)
# ==============================================================================
n_samples = len(dataset_records)

rgb_arr = np.empty(n_samples, dtype=object)
for i, r in enumerate(dataset_records):
    rgb_arr[i] = r['rgb']

wdh_arr       = np.array([[r['w_cm'], r['d_cm'], r['h_cm']] for r in dataset_records], np.float32)
ratio_arr     = np.array([[r['r_hw'], r['r_dw']] for r in dataset_records], np.float32)
bbox_wh_arr   = np.array([[r['bbox_w_px'], r['bbox_h_px']] for r in dataset_records], np.float32)
video_ids     = np.array([r['video_id'] for r in dataset_records])
track_ids     = np.array([r['tracked_id'] for r in dataset_records])
box_id_json   = np.array([r['box_id_json'] for r in dataset_records])
area_ratio    = np.array([r['area_ratio'] for r in dataset_records], np.float32)
conf_arr      = np.array([r['conf'] for r in dataset_records], np.float32)
is_clipped_arr  = np.array([r['is_clipped']  for r in dataset_records], dtype=bool)
is_max_area_arr = np.array([r['is_max_area'] for r in dataset_records], dtype=bool)
n_frames_arr  = np.array([r['n_frames_in_track'] for r in dataset_records], np.int32)
frame_idx_arr = np.array([r['frame_idx'] for r in dataset_records], np.int32)

total_bytes = sum(a.nbytes for a in rgb_arr)
print(f"rgb: object array, {n_samples}개  (총 픽셀 데이터 약 {total_bytes/1e6:.1f}MB)")
print(f"\nratio 통계:")
print(f"  r_hw (h/w): 평균={ratio_arr[:,0].mean():.3f}  표준편차={ratio_arr[:,0].std():.3f}")
print(f"  r_dw (d/w): 평균={ratio_arr[:,1].mean():.3f}  표준편차={ratio_arr[:,1].std():.3f}")
print(f"\narea_ratio 통계 (선택된 프레임들이 얼마나 큰 편인지):")
print(f"  평균={area_ratio.mean():.3f}  중앙값={np.median(area_ratio):.3f}  "
      f"최소={area_ratio.min():.3f}")
print(f"\nclipped 비율: {is_clipped_arr.mean()*100:.1f}%  "
      f"({is_clipped_arr.sum()}개 / {len(is_clipped_arr)}개)")
print(f"max_area 프레임 비율: {is_max_area_arr.mean()*100:.1f}%  "
      f"({is_max_area_arr.sum()}개 / {len(is_max_area_arr)}개, 이론상 track 수와 거의 같아야 함)")
print(f"\nvideo 수: {len(np.unique(video_ids))}개  |  샘플 수: {n_samples}개")

DATASET_OUT_PATH = os.path.join(WORK_DIR, 'BoxRegressorTuningDataYolov6.npz')
np.savez_compressed(DATASET_OUT_PATH,
    rgb=rgb_arr, wdh=wdh_arr, ratio=ratio_arr,
    bbox_wh=bbox_wh_arr, video_ids=video_ids, track_ids=track_ids,
    box_id_json=box_id_json, area_ratio=area_ratio, conf=conf_arr,
    is_clipped=is_clipped_arr, is_max_area=is_max_area_arr,
    n_frames_in_track=n_frames_arr, frame_idx=frame_idx_arr,
    allow_pickle=True)
print(f"\n저장 완료: {DATASET_OUT_PATH}")

DRIVE_DATASET_PATH = '/content/drive/MyDrive/CONTEST/cj_box_sizing/BoxRegressorTuningDataYolov6.npz'
shutil.copy(DATASET_OUT_PATH, DRIVE_DATASET_PATH)
print(f"드라이브 백업 완료: {DRIVE_DATASET_PATH}")

In [18]:
# ==============================================================================
# 데이터셋 조립 — gt_map 생성 → npz 이미지 로드(구조 확인 포함) → 함수 정의 → 실행 → 저장
#   ★ 종횡비 유지 안 함 (단순 cv2.resize)
#   ★ EXCLUDE_REMOVED 토글: True면 note=REMOVED인 (video_id, tid)는 학습 데이터에서 제외
# ==============================================================================
import numpy as np

EXCLUDE_REMOVED = True   # ★ True: REMOVED 제외 / False: REMOVED도 포함

# ── REMOVED (video_id, tracked_id) 집합 생성 ────────────────────────────────
removed_keys = set()
_removed_rows = mapping_df[mapping_df['is_removed_note']]
for _, row in _removed_rows.iterrows():
    tid_val = row['new_tracked_unique_id']
    if isinstance(tid_val, str) and tid_val.strip().upper() == 'NONE':
        continue
    removed_keys.add((str(row['video_id']), int(tid_val)))
print(f"REMOVED로 표시된 (video_id, tid) 쌍: {len(removed_keys)}개")


# ── 0. 새 데이터셋 npz 로드 + 구조 확인 + (video_id, track_id) → 이미지 매핑 생성 ─
NEW_IMG_DATASET_PATH = '/content/drive/MyDrive/CONTEST/cj_box_sizing/BoxRegressorTuningDataYolov6.npz'
_new_npz = np.load(NEW_IMG_DATASET_PATH, allow_pickle=True)

print(f"파일: {NEW_IMG_DATASET_PATH}")
print(f"키 목록: {_new_npz.files}\n")

_new_rgb        = _new_npz['rgb']            # object array, 원본 비율 (H,W,3)
_new_video_ids  = _new_npz['video_ids']
_new_track_ids  = _new_npz['track_ids']
_new_is_max     = _new_npz['is_max_area']

CROP_SIZE = 128   # 기존 video_records_tid와 동일한 입력 크기

def resize_simple(crop_rgb, target_size=CROP_SIZE):
    """★ 종횡비 유지 안 함 — 단순 정사각형 리사이즈 (기존 video_records_tid 방식과 동일)"""
    return cv2.resize(crop_rgb, (target_size, target_size)).astype(np.uint8)

# ★ is_max_area=True인 행만 골라 (video_id, tracked_id) → resize된 rgb 매핑
img_map = {}
_n_max = 0
for i in range(len(_new_video_ids)):
    if not _new_is_max[i]:
        continue
    key = (str(_new_video_ids[i]), int(_new_track_ids[i]))
    if key not in img_map:
        img_map[key] = resize_simple(_new_rgb[i], CROP_SIZE)
        _n_max += 1

print(f"새 이미지 데이터셋 로드 완료: 총 {len(_new_video_ids)}개 샘플 중 "
      f"max_area 대표 프레임 {_n_max}개 매핑")


# ── gt_map 생성: 이미 위에서 로드한 _new_npz에서 바로 생성 ──────────────
gt_map = {}
for i in range(len(_new_video_ids)):
    key = (str(_new_video_ids[i]), int(_new_track_ids[i]))
    if key not in gt_map:
        gt_map[key] = tuple(_new_npz['wdh'][i].tolist())

print(f"GT 매핑 테이블: {len(gt_map)}개 (video_id,track_id) 쌍")

# ==============================================================================
# rgb만 img_map에서 가져오도록 교체, REMOVED 필터링 추가
# ==============================================================================
def build_gt_dataset_branch1_v2():
    samples, skipped = [], []
    for vid, rec in video_records_tid.items():
        if vid not in label_map:
            skipped.append((vid, None, '라벨없음')); continue

        quad = rec['quad']
        W = rec['W']
        focal_mm = label_map[vid]['camera']['focal_length_mm']
        f_px_gt  = focal_px_of(focal_mm, W)
        f_hat_v  = f_hat_map.get(vid, FOCAL_MEAN)

        for tr in rec['tracks']:
            tid = tr['tid']

            # ★ REMOVED 필터링
            if EXCLUDE_REMOVED and (vid, tid) in removed_keys:
                skipped.append((vid, tid, 'REMOVED제외')); continue

            gt_wdh = gt_map.get((vid, tid))
            if gt_wdh is None:
                skipped.append((vid, tid, 'GT매핑없음')); continue

            img_key = (vid, tid)
            if img_key not in img_map:
                skipped.append((vid, tid, '새이미지없음')); continue
            new_rgb = img_map[img_key]

            sc_d = np.array([tr['bbox_w'], tr['bbox_h']], np.float32) * (tr['raw_med'] / 300.0)
            sc_r = (np.array([tr['bbox_w'], tr['bbox_h']], np.float32) * rail_scale_at_y(quad, tr['y_bottom'])
                   if quad is not None else None)

            samples.append({
                'video_id': vid,
                'rgb':      new_rgb,
                'depth':    tr['raw'] * (f_px_gt / 300.0),
                'raw':      tr['raw'],
                'w': gt_wdh[0], 'd': gt_wdh[1], 'h': gt_wdh[2],
                'sc_rail':  sc_r,
                'sc_da3':   sc_d,
                'f_hat':    f_hat_v,
            })

    print(f"[branch1] 샘플 {len(samples)}개  스킵 {len(skipped)}개")
    if skipped:
        reasons = {}
        for _, _, r in skipped:
            reasons[r] = reasons.get(r, 0) + 1
        print(f"  스킵 사유별 개수: {reasons}")
    return samples


def save_dataset(samples, out_path):
    ratios = [s['sc_rail'] / np.maximum(s['sc_da3'], 1e-6)
             for s in samples if s['sc_rail'] is not None]
    fallback_k = float(np.median(np.concatenate(ratios))) if ratios else 1.0

    for s in samples:
        base = s['sc_rail'] if s['sc_rail'] is not None else s['sc_da3'] * fallback_k
        s['sc'] = np.array([base[0], base[1], s['f_hat']], np.float32)

    rgb_arr   = np.stack([s['rgb'] for s in samples])
    depth_arr = np.stack([s['depth'] for s in samples])
    raw_arr   = np.stack([s['raw'] for s in samples])
    wdh_arr   = np.array([[s['w'], s['d'], s['h']] for s in samples], np.float32)
    sc_arr    = np.stack([s['sc'] for s in samples]).astype(np.float32)
    has_rail  = np.array([s['sc_rail'] is not None for s in samples])
    vid_arr   = np.array([s['video_id'] for s in samples])

    np.savez_compressed(out_path,
        rgb=rgb_arr, depth=depth_arr, raw=raw_arr,
        wdh=wdh_arr, sc=sc_arr, has_rail=has_rail, vid=vid_arr)
    print(f"저장: {out_path}  ({len(samples)}개, fallback_k={fallback_k:.4f})")


samples_b1 = build_gt_dataset_branch1_v2()
save_dataset(samples_b1, os.path.join(DATASET_OUT, 'FINAL_BOXREGRESSOR_DATA.npz'))

REMOVED로 표시된 (video_id, tid) 쌍: 12개
파일: /content/drive/MyDrive/CONTEST/cj_box_sizing/BoxRegressorTuningDataYolov6.npz
키 목록: ['rgb', 'wdh', 'ratio', 'bbox_wh', 'video_ids', 'track_ids', 'box_id_json', 'area_ratio', 'conf', 'is_clipped', 'is_max_area', 'n_frames_in_track', 'frame_idx', 'allow_pickle']

새 이미지 데이터셋 로드 완료: 총 7102개 샘플 중 max_area 대표 프레임 917개 매핑
GT 매핑 테이블: 917개 (video_id,track_id) 쌍
[branch1] 샘플 905개  스킵 18개
  스킵 사유별 개수: {'GT매핑없음': 6, 'REMOVED제외': 12}
저장: /content/work/box_dataset_with_yolo_version6/FINAL_BOXREGRESSOR_DATA.npz  (905개, fallback_k=52.2905)


# STAGE 7. Box Regressor 학습

In [21]:
# @title
# ==============================================================================
# 시드 고정 (재현성)
# ==============================================================================
import random
import torch
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ==============================================================================
# [8-1] 모델 + 학습 — GroupKFold(영상단위) 5-fold + ImageNet pretrained 백본
#   ★ FINAL_BOXREGRESSOR_DATA.npz 사용, 각 fold epochs=100 고정
# ==============================================================================
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from sklearn.model_selection import GroupKFold
import time, json

device = 'cuda' if torch.cuda.is_available() else 'cpu'

_d = np.load(os.path.join(DATASET_OUT, 'FINAL_BOXREGRESSOR_DATA.npz'))
rgb, depth, wdh, sc, vid_arr = _d['rgb'], _d['depth'], _d['wdh'], _d['sc'], _d['vid']
print(f"샘플 {len(rgb)}  wdh평균 {wdh.mean(0).round(1)}  sc평균 {sc.mean(0).round(2)}")

DEPTH_MEAN, DEPTH_STD = float(depth.mean()), float(depth.std() + 1e-6)
SC_MEAN,    SC_STD    = sc.mean(0),  sc.std(0)  + 1e-6

SC_NOISE = 0.05
F_NOISE  = 0.15

den   = np.maximum(np.stack([sc[:, 0], sc[:, 1], sc[:, 1]], 1), 1e-3)
ratio = wdh / den

R_MEAN, R_STD = ratio.mean(0), ratio.std(0) + 1e-6
R_CLIP_LO, R_CLIP_HI = 0.15, 3.0
print(f"비율 분포: 평균 {ratio.mean(0).round(3)}  std {ratio.std(0).round(3)}")


class BoxDataset(Dataset):
    def __init__(self, idx, train=True):
        self.idx = idx; self.train = train
    def __len__(self): return len(self.idx)
    def __getitem__(self, k):
        i = self.idx[k]
        r  = rgb[i].astype(np.float32) / 255.0
        dp = (depth[i] - DEPTH_MEAN) / DEPTH_STD
        if self.train and np.random.rand() < 0.5: r = r[:, ::-1].copy(); dp = dp[:, ::-1].copy()
        img = np.concatenate([r.transpose(2, 0, 1), dp[None]], 0)
        sc_i = sc[i].copy()
        if self.train:
            sc_i[:2] = sc_i[:2] * (1.0 + np.random.randn(2).astype(np.float32) * SC_NOISE)
            sc_i[2]  = sc_i[2]  * (1.0 + np.random.randn() * F_NOISE)
        sc_n = (sc_i - SC_MEAN) / SC_STD
        y = (ratio[i] - R_MEAN) / R_STD
        return (torch.from_numpy(img).float(),
                torch.from_numpy(sc_n).float(),
                torch.from_numpy(y).float())


class BoxRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        old_w = m.conv1.weight.data.clone()
        m.conv1 = nn.Conv2d(4, 64, 7, 2, 3, bias=False)
        with torch.no_grad():
            m.conv1.weight[:, :3] = old_w
            m.conv1.weight[:, 3:] = old_w.mean(dim=1, keepdim=True)
        self.backbone = nn.Sequential(*list(m.children())[:-1])
        self.head = nn.Sequential(
            nn.Linear(512 + 3, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 3))
    def forward(self, img, sc):
        f = self.backbone(img).flatten(1)
        return self.head(torch.cat([f, sc], 1))


def eval_mae(model, val_idx):
    model.eval(); errs = []
    with torch.no_grad():
        for bi in range(0, len(val_idx), 64):
            idx = np.array(val_idx[bi:bi+64])
            imgs = np.stack([np.concatenate(
                [rgb[i].astype(np.float32).transpose(2, 0, 1) / 255.0,
                 ((depth[i] - DEPTH_MEAN) / DEPTH_STD)[None]], 0) for i in idx])
            scs = ((sc[idx] - SC_MEAN) / SC_STD).astype(np.float32)
            r_pr = model(torch.from_numpy(imgs).float().to(device),
                         torch.from_numpy(scs).to(device)).cpu().numpy() * R_STD + R_MEAN
            r_pr = np.clip(r_pr, R_CLIP_LO, R_CLIP_HI)
            errs.append(np.abs(r_pr * den[idx] - wdh[idx]))
    e = np.concatenate(errs); return e.mean(), e.mean(0)


# ──────────────────────────────────────────────────────────────────────────────
# ★ 파라미터 ★
# ──────────────────────────────────────────────────────────────────────────────
N_FOLDS  = 5
N_EPOCHS = 100
CHECK_EVERY = 5
# ──────────────────────────────────────────────────────────────────────────────

gkf = GroupKFold(n_splits=N_FOLDS)
indices = np.arange(len(rgb))

fold_results = []

for fold_num, (tr_idx, val_idx) in enumerate(gkf.split(indices, groups=vid_arr)):
    tr_idx, val_idx = tr_idx.tolist(), val_idx.tolist()
    assert not (set(vid_arr[tr_idx]) & set(vid_arr[val_idx])), "영상 누수!"

    torch.manual_seed(42); np.random.seed(42)
    model = BoxRegressor().to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPOCHS)
    crit  = nn.L1Loss()
    tr_loader = DataLoader(BoxDataset(tr_idx, True), batch_size=32, shuffle=True, num_workers=2)

    BEST_PT = os.path.join(DATASET_OUT, f'best_kfold{fold_num}_rgbfix_pretrained_ep100.pt')
    best, best_ep = 1e9, 0

    print(f"\n=== Fold {fold_num+1}/{N_FOLDS} === "
          f"train {len(tr_idx)}샘플({len(set(vid_arr[tr_idx]))}영상)  "
          f"val {len(val_idx)}샘플({len(set(vid_arr[val_idx]))}영상)")

    t0 = time.time()
    for ep in range(N_EPOCHS):
        model.train()
        for img, s, y in tr_loader:
            img, s, y = img.to(device), s.to(device), y.to(device)
            opt.zero_grad(); loss = crit(model(img, s), y); loss.backward(); opt.step()
        sched.step()

        if (ep + 1) % CHECK_EVERY == 0 or ep == 0:
            mae, per = eval_mae(model, val_idx)
            marker = ""
            if mae < best:
                best, best_ep = mae, ep + 1
                torch.save(model.state_dict(), BEST_PT)
                marker = "  ★"
            print(f"  [fold{fold_num}] ep{ep+1:3d} val MAE={mae:.3f} "
                  f"(w{per[0]:.2f} d{per[1]:.2f} h{per[2]:.2f}){marker}")

    fold_results.append({'fold': fold_num, 'best_mae': float(best), 'best_ep': int(best_ep)})
    print(f"  → fold{fold_num} 최고 val MAE {best:.3f}cm (ep{best_ep})  "
          f"총 소요 {time.time()-t0:.0f}s")

# ==============================================================================
# 종합 결과
# ==============================================================================
maes     = [r['best_mae'] for r in fold_results]
best_eps = [r['best_ep']  for r in fold_results]

print(f"\n{'='*60}")
print(f"[영상단위 5-fold + pretrained, epochs={N_EPOCHS}, FINAL_BOXREGRESSOR_DATA]")
print(f"fold별 결과: {[round(m,3) for m in maes]}")
print(f"평균 val MAE: {np.mean(maes):.3f}cm  ±{np.std(maes):.3f}")
print(f"fold별 최적 epoch: {best_eps}")

with open(os.path.join(DATASET_OUT, 'kfold5_ep100_results.json'), 'w') as f:
    json.dump(fold_results, f, indent=2)
print(f"결과 저장: kfold5_ep100_results.json")

샘플 905  wdh평균 [26.6 27.  14.1]  sc평균 [29.92 25.37 10.26]
비율 분포: 평균 [0.882 1.11  0.583]  std [0.09  0.431 0.2  ]
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 221MB/s]



=== Fold 1/5 === train 724샘플(80영상)  val 181샘플(20영상)
  [fold0] ep  1 val MAE=5.467 (w5.78 d6.92 h3.70)  ★
  [fold0] ep  5 val MAE=2.669 (w0.93 d4.48 h2.59)  ★
  [fold0] ep 10 val MAE=1.865 (w0.91 d3.00 h1.69)  ★
  [fold0] ep 15 val MAE=1.926 (w0.77 d3.23 h1.78)
  [fold0] ep 20 val MAE=1.866 (w0.77 d3.08 h1.74)
  [fold0] ep 25 val MAE=1.863 (w0.75 d3.05 h1.79)  ★
  [fold0] ep 30 val MAE=1.765 (w0.79 d2.85 h1.66)  ★
  [fold0] ep 35 val MAE=1.787 (w0.75 d2.96 h1.66)
  [fold0] ep 40 val MAE=1.668 (w0.73 d2.56 h1.71)  ★
  [fold0] ep 45 val MAE=1.820 (w0.83 d2.79 h1.83)
  [fold0] ep 50 val MAE=1.724 (w0.71 d2.75 h1.71)
  [fold0] ep 55 val MAE=1.681 (w0.65 d2.74 h1.65)
  [fold0] ep 60 val MAE=1.655 (w0.67 d2.68 h1.62)  ★
  [fold0] ep 65 val MAE=1.661 (w0.66 d2.68 h1.64)
  [fold0] ep 70 val MAE=1.692 (w0.65 d2.80 h1.62)
  [fold0] ep 75 val MAE=1.563 (w0.62 d2.52 h1.54)  ★
  [fold0] ep 80 val MAE=1.627 (w0.65 d2.61 h1.63)
  [fold0] ep 85 val MAE=1.631 (w0.67 d2.58 h1.64)
  [fold0] ep 90 val MAE

# norm_stats.json 만들기

In [23]:
# Colab에서 한 번만 실행 — 정규화 상수 + f̂ 회귀계수 저장
import json

norm_stats = {
    "DEPTH_MEAN": float(DEPTH_MEAN),
    "DEPTH_STD": float(DEPTH_STD),
    "SC_MEAN": SC_MEAN.tolist(),
    "SC_STD": SC_STD.tolist(),
    "R_MEAN": R_MEAN.tolist(),
    "R_STD": R_STD.tolist(),
    "R_CLIP_LO": float(R_CLIP_LO),
    "R_CLIP_HI": float(R_CLIP_HI),
    "SC_FALLBACK_K": float(SC_FALLBACK_K),
    "FOCAL_MEAN": float(FOCAL_MEAN),
    "F_COEF": F_COEF,      # 이미 list
    "F_INT": float(F_INT),
    "SENSOR_W": float(SENSOR_W),
    "RAIL_WIDTH_CM": float(RAIL_WIDTH_CM),
    "CROP_SIZE": int(CROP_SIZE),
}

with open('/content/drive/MyDrive/CONTEST/cj_box_sizing/norm_stats.json', 'w') as f:
    json.dump(norm_stats, f, indent=2)
print("✅ norm_stats.json 저장 완료")

✅ norm_stats.json 저장 완료


# ONNX 변환

In [24]:
!pip install -q onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 126.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 17.6 MB/s eta 0:00:00


In [25]:
# ==============================================================================
# [8-2] K-fold 5개 BoxRegressor 모두 ONNX 변환
# ==============================================================================
import torch

ONNX_OUT_DIR = PROJECT_DIR
os.makedirs(ONNX_OUT_DIR, exist_ok=True)

dummy_img = torch.randn(1, 4, CROP_SIZE, CROP_SIZE, device=device)
dummy_sc  = torch.randn(1, 3, device=device)

for fold_num in range(N_FOLDS):
    BEST_PT_fold = os.path.join(DATASET_OUT, f'best_kfold{fold_num}_rgbfix_pretrained_ep100.pt')
    ONNX_OUT_PATH = os.path.join(ONNX_OUT_DIR, f'box_regressor_fold{fold_num}.onnx')

    model.load_state_dict(torch.load(BEST_PT_fold, map_location=device))
    model.eval()

    torch.onnx.export(
        model,
        (dummy_img, dummy_sc),
        ONNX_OUT_PATH,
        input_names=['img', 'sc'],
        output_names=['ratio_pred'],
        dynamic_axes={
            'img': {0: 'batch_size'},
            'sc': {0: 'batch_size'},
            'ratio_pred': {0: 'batch_size'},
        },
        opset_version=17,
        do_constant_folding=True,
    )
    print(f"✅ fold{fold_num} ONNX 저장 완료: {ONNX_OUT_PATH}")

/tmp/ipykernel_2496/3221995901.py:19: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0715 09:44:08.428000 2496 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `BoxRegressor([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `BoxRegressor([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...


/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/exporter/_onnx_program.py:487: UserWarning: # The axis name: batch_size will not be used, since it shares the same shape constraints with another axis: batch_size.
  rename_mapping = _dynamic_shapes.create_rename_mapping(
/tmp/ipykernel_2496/3221995901.py:19: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0715 09:44:13.894000 2496 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider s

...

[torch.onnx] Optimize the ONNX graph... ✅
✅ fold3 ONNX 저장 완료: /content/drive/MyDrive/CONTEST/cj_box_sizing/box_regressor_fold3.onnx
[torch.onnx] Obtain model graph for `BoxRegressor([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `BoxRegressor([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...


/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/exporter/_onnx_program.py:487: UserWarning: # The axis name: batch_size will not be used, since it shares the same shape constraints with another axis: batch_size.
  rename_mapping = _dynamic_shapes.create_rename_mapping(


[torch.onnx] Optimize the ONNX graph... ✅
✅ fold4 ONNX 저장 완료: /content/drive/MyDrive/CONTEST/cj_box_sizing/box_regressor_fold4.onnx


# Requirements.txt 만들기

In [26]:
# ==============================================================================
# requirements.txt 생성 (전체 pip freeze)
# ==============================================================================
import subprocess

req_path = "/content/drive/MyDrive/CONTEST/cj_box_sizing/requirements_Training_BoxRegressor.txt"

result = subprocess.run(["pip", "freeze"], capture_output=True, text=True)

with open(req_path, "w") as f:
    f.write(result.stdout)

print(f"총 {len(result.stdout.splitlines())}개 패키지 저장 완료: {req_path}")
print("\n--- 미리보기 (상위 20줄) ---")
print("\n".join(result.stdout.splitlines()[:20]))

총 713개 패키지 저장 완료: /content/drive/MyDrive/CONTEST/cj_box_sizing/requirements_Training_BoxRegressor.txt

--- 미리보기 (상위 20줄) ---
absl-py==1.4.0
accelerate==1.14.0
access==1.1.10.post3
affine==2.4.0
aiofiles==25.1.0
aiohappyeyeballs==2.6.2
aiohttp==3.14.1
aiosignal==1.4.0
aiosqlite==0.22.1
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.12.0
altair==5.5.0
annotated-doc==0.0.4
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.14.0
anywidget==0.9.21
apsw==3.53.2.0
